# LangGraph Agentic RAG — SEC EDGAR v2

**Changes from v1:**
- **Dataset**: Uses local `sec_chunks.jsonl` (12 companies, 155 filings, 2023–2026) instead of FinanceBench
- **Proposal 1 — Query Decomposition**: A `Planner` node decomposes multi-period / cross-company questions into 2–3 sub-queries, each with optional `ticker`/`filing_year` metadata filters for targeted retrieval
- **Proposal 2 — Refined Critic**: `CriticOutput.decision` now has a third state `insufficient_data` — the agent safely exits when required data is absent from the filing rather than looping

In [1]:
import os
import re
import time
import json
import random
import warnings
import threading
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, TypedDict, Literal, Tuple
from collections import deque, Counter

import chromadb
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder

from pydantic import BaseModel, Field, field_validator, model_validator

from langgraph.graph import StateGraph, END
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq
from langchain_google_genai import ChatGoogleGenerativeAI

warnings.filterwarnings("ignore")

c:\Users\jeeey\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [2]:
if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = input("Enter your GROQ_API_KEY (leave blank if using Gemini): ").strip()

if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = input("Enter your GOOGLE_API_KEY (leave blank if using Groq): ").strip()

GROQ_API_KEY   = os.environ.get("GROQ_API_KEY", "")
GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY", "")

In [3]:
CONFIG: Dict[str, Any] = {
    "random_seed": 42,

    # ── Pilot vs full run ────────────────────────────────────────────────────
    "use_pilot":         False,   # True = pilot (10 questions); False = full (80 questions)
    "pilot_n_questions": 10,
    "full_n_questions":  80,
    "use_llm_judge": False,
    "use_few_shot_examples": True,
    "pilot_judge_sample_n": 1,
    "full_judge_sample_n":  2,

    # 'dev' (cheaper/faster) or 'final' (best quality)
    "execution_profile": "dev",

    # SEC dataset paths
    "sec_chunks_path": r"C:\Users\jeeey\GenAI-with-LLMs\sec_rag_team_share\sec_data\chunks\sec_chunks.jsonl",
    "chroma_db_path":  r"C:\Users\jeeey\GenAI-with-LLMs\sec_rag_team_share\chroma_db",
    "sec_eval_csv_path": r"C:\Users\jeeey\GenAI-with-LLMs\sec_rag_team_share\evaluation\GenAI Eval QA.csv",
    "results_input_excel_path": r"C:\Users\jeeey\GenAI-with-LLMs\LangGraph\eval_skeleton (LangGraph).xlsx",
    "train_split": "dev",
    "eval_split":  "test",
    "auto_export_results_input": True,

    # retrieval / agent loop controls
    "max_context_retries": 1,
    "max_repair_retrievals": 1,
    "bm25_top_k": 8,
    "dense_top_k": 8,
    "rerank_top_k": 5,

    # decomposition: chunks retrieved per sub-query before merging
    "decomposition_top_k_per_subquery": 4,

    # Embedding model MUST match what was used to build the ChromaDB collection (768-dim)
    "dense_model_name":    "sentence-transformers/all-mpnet-base-v2",
    "reranker_model_name": "cross-encoder/ms-marco-MiniLM-L-6-v2",

    # Provider switch: 'groq' or 'gemini'
    "provider": "groq",

    # Groq models
    "groq_dev_generator_model":   "llama-3.1-8b-instant",
    "groq_dev_agent_model":       "llama-3.1-8b-instant",
    "groq_dev_judge_model":       "llama-3.1-8b-instant",
    "groq_final_generator_model": "meta-llama/llama-4-scout-17b-16e-instruct",
    "groq_final_agent_model":     "llama-3.1-8b-instant",
    "groq_final_judge_model":     "llama-3.1-8b-instant",
    "groq_fallback_agent_models": ["llama-3.1-8b-instant", "qwen/qwen3-32b"],
    "groq_fallback_generator_models": ["llama-3.1-8b-instant", "qwen/qwen3-32b"],
    "groq_fallback_judge_models": ["llama-3.1-8b-instant", "qwen/qwen3-32b"],

    # Gemini models
    "gemini_dev_generator_model":   "gemini-2.0-flash-lite",
    "gemini_dev_agent_model":       "gemini-2.0-flash-lite",
    "gemini_dev_judge_model":       "gemini-2.0-flash-lite",
    "gemini_final_generator_model": "gemini-2.0-flash",
    "gemini_final_agent_model":     "gemini-2.0-flash",
    "gemini_final_judge_model":     "gemini-2.0-flash",

    # temperatures
    "temperature_planner":      0.0,
    "temperature_rewriter":     0.2,
    "temperature_context_eval": 0.0,
    "temperature_generator":    0.2,
    "temperature_critic":       0.0,
    "temperature_repair":       0.1,
    "temperature_judge":        0.0,
    # context formatting caps to reduce token usage in control nodes
    "generator_max_context_chunks": 8,
    "generator_max_context_chars": 12000,
    "control_max_context_chunks": 4,
    "control_max_context_chars": 6000,
    "judge_max_context_chunks": 3,
    "judge_max_context_chars": 4000,
    "enable_retrieval_sanity_check": True,

    # rate limiting — stay slightly under Groq's hard 30 RPM cap
    "max_rpm": 28,

    # inter-question sleep
    "inter_question_sleep_sec": 0.5,

    # retry settings
    "llm_max_retries": 3,
    "llm_retry_base_delay_sec": 5,

    # Optional LLM judge sample size (per pipeline) — smaller for pilot
}

CONFIG["max_rpm"] = 28 if CONFIG["provider"] == "groq" else 10
CONFIG["judge_sample_n"] = (
    CONFIG["pilot_judge_sample_n"] if CONFIG["use_pilot"] else CONFIG["full_judge_sample_n"]
) if CONFIG["use_llm_judge"] else 0


def resolve_model_name(role: str) -> str:
    provider = CONFIG["provider"]
    profile  = CONFIG["execution_profile"]
    return CONFIG[f"{provider}_{profile}_{role}_model"]


def resolve_fallback_model_names(role: str) -> List[str]:
    provider = CONFIG["provider"]
    primary = resolve_model_name(role)
    fallbacks = CONFIG.get(f"{provider}_fallback_{role}_models", [])
    ordered = [primary] + list(fallbacks)
    return list(dict.fromkeys([m for m in ordered if m]))

In [4]:
# ---------------------------------------------------------------------------
# Evaluation dataset from sec_rag_team_share/evaluation/GenAI Eval QA.csv
# Use the 'dev' split for prompt development / training-style iteration and
# the 'test' split for evaluation.
# ---------------------------------------------------------------------------

raw_eval_df = pd.read_csv(CONFIG["sec_eval_csv_path"])
raw_eval_df = raw_eval_df[raw_eval_df["question_id"].notna()].copy()
raw_eval_df["question_id"] = raw_eval_df["question_id"].astype(str).str.strip()
raw_eval_df = raw_eval_df[raw_eval_df["question_id"] != ""].copy()

full_eval_df = raw_eval_df.rename(columns={
    "question_id": "id",
    "expected_answer": "reference_answer",
}).copy()

full_eval_df["question_number"] = pd.to_numeric(full_eval_df["id"], errors="coerce")
full_eval_df["id"] = full_eval_df["id"].apply(lambda x: f"SEC_{int(float(x)):03d}" if str(x).replace(".", "", 1).isdigit() else str(x))
full_eval_df["expected_decision"] = full_eval_df["expected_decision"].fillna("ANSWER").astype(str).str.strip().str.lower()
full_eval_df["question_type"] = full_eval_df["question_type"].fillna("extractive").astype(str).str.strip()
full_eval_df["company"] = full_eval_df["company"].fillna("").astype(str).str.strip()
full_eval_df["reference_answer"] = full_eval_df["reference_answer"].fillna("").astype(str).str.strip()
full_eval_df["split"] = full_eval_df["split"].fillna("").astype(str).str.strip().str.lower()

train_df = full_eval_df[full_eval_df["split"] == CONFIG["train_split"]].reset_index(drop=True)
eval_source_df = full_eval_df[full_eval_df["split"] == CONFIG["eval_split"]].reset_index(drop=True)

if CONFIG["use_pilot"]:
    eval_df = eval_source_df.sample(
        n=min(CONFIG["pilot_n_questions"], len(eval_source_df)),
        random_state=CONFIG["random_seed"],
    ).sort_values("id").reset_index(drop=True)
    print(f"PILOT RUN: {len(eval_df)} questions from split='{CONFIG['eval_split']}'")
else:
    eval_df = eval_source_df.sample(
        n=min(CONFIG["full_n_questions"], len(eval_source_df)),
        random_state=CONFIG["random_seed"],
    ).sort_values("id").reset_index(drop=True)
    print(f"FULL RUN: {len(eval_df)} questions from split='{CONFIG['eval_split']}'")

print(f"Training/dev split size: {len(train_df)} | Evaluation/test split size: {len(eval_source_df)}")
print(eval_df[["id", "question_type", "company", "ticker", "form_type", "filing_year"]].to_string(index=False))

FULL RUN: 80 questions from split='test'
Training/dev split size: 20 | Evaluation/test split size: 80
     id question_type            company ticker form_type  filing_year
SEC_006    extractive             NVIDIA   NVDA      10-K       2024.0
SEC_007    extractive             NVIDIA   NVDA      10-K       2025.0
SEC_008    extractive     JPMorgan Chase    JPM      10-K       2023.0
SEC_009    extractive     JPMorgan Chase    JPM      10-K       2024.0
SEC_010    extractive    Bank of America    BAC      10-Q       2025.0
SEC_011    extractive    Bank of America    BAC      10-K       2024.0
SEC_012    extractive Berkshire Hathaway  BRK-B      10-K       2025.0
SEC_013    extractive Berkshire Hathaway  BRK-B      10-Q       2023.0
SEC_014    extractive  Johnson & Johnson    JNJ      10-Q       2025.0
SEC_015    extractive  Johnson & Johnson    JNJ      10-K       2025.0
SEC_016    extractive UnitedHealth Group    UNH      10-Q       2024.0
SEC_017    extractive UnitedHealth Group    UN

In [5]:
# ---------------------------------------------------------------------------
# SEC corpus loading
# ---------------------------------------------------------------------------

def load_sec_chunks(path: str) -> pd.DataFrame:
    """Load sec_chunks.jsonl and filter out low-value chunks."""
    chunks = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                chunks.append(json.loads(line))
    df = pd.DataFrame(chunks)
    before = len(df)
    df = df[~df["flag_low_value_combined"].fillna(False)].reset_index(drop=True)
    print(
        f"Loaded {before} total chunks → {len(df)} usable after filtering low-value "
        f"({before - len(df)} removed)"
    )
    print(
        f"Companies: {sorted(df['ticker'].unique())}  |  "
        f"Filing years: {sorted(df['filing_year'].unique())}"
    )
    return df


def sec_df_to_chunk_dicts(df: pd.DataFrame) -> List[Dict[str, Any]]:
    """Convert SEC DataFrame to the chunk dict format expected by CorpusIndex."""
    chunks = []
    for i, row in df.iterrows():
        doc_name = f"{row['ticker']}_{row['form_type']}_{str(row['filing_date'])[:10]}"
        contextual_text = (
            f"Company: {row['company_name']} ({row['ticker']})\n"
            f"Filing: {row['form_type']} | Date: {str(row['filing_date'])[:10]} | Year: {row['filing_year']}\n"
            f"Section: {row['section_title']}\n"
            f"Content: {row['text']}"
        )
        chunks.append({
            "chunk_id":     i,                  # integer row index (BM25 lookup)
            "chunk_id_str": str(row["chunk_id"]), # original string ID (ChromaDB cross-ref)
            "ticker":       row["ticker"],
            "company":      row["company_name"],
            "doc_name":     doc_name,
            "filing_year":  int(row["filing_year"]),
            "form_type":    row["form_type"],
            "page_num":     int(row.get("chunk_index", 0)),
            "raw_chunk":    row["text"],
            "contextual_chunk": contextual_text,
        })
    return chunks


print("Loading SEC chunks…")
sec_df = load_sec_chunks(CONFIG["sec_chunks_path"])

Loading SEC chunks…
Loaded 13152 total chunks → 12606 usable after filtering low-value (546 removed)
Companies: ['AAPL', 'BAC', 'BRK-B', 'COST', 'CVX', 'JNJ', 'JPM', 'MSFT', 'NVDA', 'UNH', 'WMT', 'XOM']  |  Filing years: [np.int64(2023), np.int64(2024), np.int64(2025)]


In [6]:
# dense_model: encodes queries at search time (single vector — fast)
# ChromaDB holds the pre-built document embeddings so no all-chunk encoding needed
print("Loading dense embedder (query-time use only)…")
dense_model = SentenceTransformer(CONFIG["dense_model_name"])
print("Loading reranker…")
reranker = CrossEncoder(CONFIG["reranker_model_name"])
print("Models ready.")

Loading dense embedder (query-time use only)…


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading reranker…


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Models ready.


In [7]:
# ---------------------------------------------------------------------------
# RetrievedChunk + CorpusIndex
#
# Dense search: ChromaDB (pre-built, loaded in ~1s — no re-encoding at startup)
# Keyword search: BM25Okapi (built from text in-memory, fast)
# Reranking: CrossEncoder (query-time, on merged candidates)
#
# Metadata filtering (ticker, filing_year, form_type) applies to BOTH:
#   BM25  → numpy score mask on self.df
#   Dense → ChromaDB `where` clause pushed to the vector store
# ---------------------------------------------------------------------------

@dataclass
class RetrievedChunk:
    doc_name: str
    company: str
    page_num: int
    chunk_id: str       # string chunk_id — consistent key for BM25 + ChromaDB dedup
    raw_chunk: str
    contextual_chunk: str
    score: float
    source: str


def _rrf_score(rank: int, k: int = 60) -> float:
    return 1.0 / (k + rank + 1)


class CorpusIndex:
    def __init__(self, chunks: List[Dict[str, Any]], chroma_path: str):
        self.df = pd.DataFrame(chunks).copy()

        # BM25 — built from contextual text, fast (no ML)
        print("  Building BM25 index…")
        self.df["bm25_tokens"] = self.df["contextual_chunk"].str.lower().str.split()
        self.bm25 = BM25Okapi(self.df["bm25_tokens"].tolist())

        # String chunk_id → df integer row index (for BM25 chunk materialisation)
        self._str_to_row: Dict[str, int] = {
            row["chunk_id_str"]: idx for idx, row in self.df.iterrows()
        }

        # ChromaDB — load pre-built collection (embeddings already stored)
        print("  Loading ChromaDB collection…")
        client = chromadb.PersistentClient(path=chroma_path)
        collections = client.list_collections()
        if not collections:
            raise RuntimeError(f"No ChromaDB collections found at {chroma_path!r}")
        self.chroma_col = client.get_collection(collections[0].name)
        print(
            f"  ChromaDB ready: '{collections[0].name}'  "
            f"({self.chroma_col.count():,} docs)"
        )
        print(f"  CorpusIndex ready: {len(self.df):,} BM25 chunks + ChromaDB dense index")

    # ── Chunk materialisation helpers ────────────────────────────────────────

    def _chunk_from_row(self, row_idx: int, score: float, source: str) -> RetrievedChunk:
        row = self.df.iloc[row_idx]
        return RetrievedChunk(
            doc_name=row["doc_name"],
            company=row["company"],
            page_num=int(row.get("page_num", 0)),
            chunk_id=row["chunk_id_str"],
            raw_chunk=row["raw_chunk"],
            contextual_chunk=row["contextual_chunk"],
            score=float(score),
            source=source,
        )

    @staticmethod
    def _contextual_from_meta(text: str, meta: dict) -> str:
        return (
            f"Company: {meta['company_name']} ({meta['ticker']})\n"
            f"Filing: {meta['form_type']} | Date: {str(meta['filing_date'])[:10]} "
            f"| Year: {meta['filing_year']}\n"
            f"Section: {meta['section_title']}\n"
            f"Content: {text}"
        )

    def _chunk_from_chroma(self, doc_id: str, text: str, meta: dict, dist: float) -> RetrievedChunk:
        doc_name = f"{meta['ticker']}_{meta['form_type']}_{str(meta['filing_date'])[:10]}"
        return RetrievedChunk(
            doc_name=doc_name,
            company=meta["company_name"],
            page_num=int(meta.get("chunk_index", 0)),
            chunk_id=doc_id,                              # same string as BM25 chunk_id_str
            raw_chunk=text,
            contextual_chunk=self._contextual_from_meta(text, meta),
            score=1.0 - float(dist),                      # cosine distance → similarity
            source="chroma_dense",
        )

    # ── Metadata filter builders ─────────────────────────────────────────────

    def _bm25_mask(
        self, ticker: str = None, filing_year: int = None, form_type: str = None
    ) -> Optional[np.ndarray]:
        """Float mask (1.0/0.0) for BM25 score array. None = no filter."""
        if not ticker and not filing_year and not form_type:
            return None
        mask = np.ones(len(self.df), dtype=float)
        if ticker:
            mask *= (self.df["ticker"].str.upper() == ticker.upper()).astype(float).values
        if filing_year:
            mask *= (self.df["filing_year"] == int(filing_year)).astype(float).values
        if form_type:
            mask *= (self.df["form_type"].str.upper() == form_type.upper()).astype(float).values
        if mask.sum() < 5:
            print(f"  [Warn] BM25 filter matched < 5 chunks — using full corpus")
            return None
        return mask

    def _chroma_where(
        self, ticker: str = None, filing_year: int = None, form_type: str = None
    ) -> Optional[dict]:
        """ChromaDB `where` clause for metadata filtering. None = no filter."""
        conditions = []
        if ticker:
            conditions.append({"ticker": {"$eq": ticker.upper()}})
        if filing_year:
            conditions.append({"filing_year": {"$eq": int(filing_year)}})
        if form_type:
            conditions.append({"form_type": {"$eq": form_type.upper()}})
        if not conditions:
            return None
        return conditions[0] if len(conditions) == 1 else {"$and": conditions}

    # ── Search methods ────────────────────────────────────────────────────────

    def bm25_search(
        self, query: str, top_k: int, mask: Optional[np.ndarray] = None
    ) -> List[RetrievedChunk]:
        scores = np.array(self.bm25.get_scores(query.lower().split()))
        if mask is not None:
            scores = scores * mask
        top_idx = np.argsort(scores)[::-1][:top_k]
        return [self._chunk_from_row(i, scores[i], "bm25") for i in top_idx if scores[i] > 0]

    def dense_search(
        self,
        query: str,
        top_k: int,
        ticker: str = None,
        filing_year: int = None,
        form_type: str = None,
    ) -> List[RetrievedChunk]:
        """ANN search via ChromaDB using a single query embedding (fast)."""
        q_emb = dense_model.encode([query], normalize_embeddings=True)[0].tolist()
        where = self._chroma_where(ticker, filing_year, form_type)
        kwargs: dict = {
            "query_embeddings": [q_emb],
            "n_results": top_k,
            "include": ["documents", "metadatas", "distances"],
        }
        if where:
            kwargs["where"] = where
        results = self.chroma_col.query(**kwargs)
        return [
            self._chunk_from_chroma(doc_id, text, meta, dist)
            for doc_id, text, meta, dist in zip(
                results["ids"][0],
                results["documents"][0],
                results["metadatas"][0],
                results["distances"][0],
            )
        ]

    def hybrid_search(
        self,
        query: str,
        bm25_top_k: int,
        dense_top_k: int,
        rerank_top_k: int,
        ticker: str = None,
        filing_year: int = None,
        form_type: str = None,
    ) -> List[RetrievedChunk]:
        mask         = self._bm25_mask(ticker, filing_year, form_type)
        bm25_results  = self.bm25_search(query, bm25_top_k, mask)
        dense_results = self.dense_search(query, dense_top_k, ticker, filing_year, form_type)

        # RRF merge — deduplicate on string chunk_id
        rrf: Dict[str, float] = {}
        pool: Dict[str, RetrievedChunk] = {}
        for rank, c in enumerate(bm25_results):
            rrf[c.chunk_id]  = rrf.get(c.chunk_id, 0.0) + _rrf_score(rank)
            pool[c.chunk_id] = c
        for rank, c in enumerate(dense_results):
            rrf[c.chunk_id] = rrf.get(c.chunk_id, 0.0) + _rrf_score(rank)
            if c.chunk_id not in pool:
                pool[c.chunk_id] = c

        candidates = [pool[k] for k in sorted(rrf, key=rrf.__getitem__, reverse=True)]
        if not candidates:
            return []

        # CrossEncoder rerank
        scores = reranker.predict(
            [(query, c.contextual_chunk) for c in candidates], show_progress_bar=False
        )
        for c, s in zip(candidates, scores):
            c.score  = float(s)
            c.source = "hybrid_reranked"

        return sorted(candidates, key=lambda x: x.score, reverse=True)[:rerank_top_k]


# ---------------------------------------------------------------------------
# Build global index — BM25 + ChromaDB load (~5s, no GPU/encoding needed)
# ---------------------------------------------------------------------------
print("Building global corpus index…")
chunk_dicts  = sec_df_to_chunk_dicts(sec_df)
global_index = CorpusIndex(chunk_dicts, chroma_path=CONFIG["chroma_db_path"])
print("Done.")

Building global corpus index…
  Building BM25 index…
  Loading ChromaDB collection…
  ChromaDB ready: 'sec_filings'  (12,606 docs)
  CorpusIndex ready: 12,606 BM25 chunks + ChromaDB dense index
Done.


In [8]:
def make_llm(model_name: str, temperature: float):
    if CONFIG["provider"] == "groq":
        return ChatGroq(model=model_name, temperature=temperature, groq_api_key=GROQ_API_KEY)
    elif CONFIG["provider"] == "gemini":
        return ChatGoogleGenerativeAI(model=model_name, temperature=temperature, google_api_key=GOOGLE_API_KEY)
    else:
        raise ValueError(f"Unknown provider: {CONFIG['provider']!r}")


def resolve_role_temperature(role: str, task_name: str = None) -> float:
    if task_name is not None:
        key = f"temperature_{task_name}"
        if key in CONFIG:
            return CONFIG[key]
    return CONFIG[f"temperature_{role}"] if f"temperature_{role}" in CONFIG else 0.0


def get_llm_candidates(role: str, task_name: str = None):
    temperature = resolve_role_temperature(role, task_name)
    return [
        (model_name, make_llm(model_name, temperature))
        for model_name in resolve_fallback_model_names(role)
    ]


_preferred_models_by_task: Dict[str, str] = {}


def order_model_candidates(role: str, task_name: str = None):
    candidates = get_llm_candidates(role, task_name)
    preference_key = task_name or role
    preferred = _preferred_models_by_task.get(preference_key)
    if not preferred:
        return candidates
    candidate_names = [name for name, _ in candidates]
    if preferred not in candidate_names:
        return candidates
    preferred_idx = candidate_names.index(preferred)
    return candidates[preferred_idx:]

In [9]:
# ---------------------------------------------------------------------------
# Pydantic output schemas
# ---------------------------------------------------------------------------

def _coerce_bool(v) -> bool:
    if isinstance(v, str):
        return v.strip().lower() in ("true", "1", "yes")
    return bool(v)

def _coerce_float(v) -> float:
    if isinstance(v, str):
        return float(v.strip())
    return float(v)


# ── Proposal 1: Query Decomposition ─────────────────────────────────────────

class SubQuery(BaseModel):
    query: Optional[str] = Field(default=None, description="Search-optimized sub-query text for retrieval")
    ticker: Optional[str] = Field(
        default=None,
        description="Company ticker (e.g. AAPL, MSFT) if this sub-query is specific to one company. "
                    "None for cross-company sub-queries."
    )
    filing_year: Optional[int] = Field(
        default=None,
        description="Specific fiscal year (e.g. 2024) if this sub-query is year-specific. None otherwise."
    )
    form_type: Optional[str] = Field(
        default=None,
        description="10-K or 10-Q if the sub-query is specific to one form type. None otherwise."
    )

    @field_validator("query", mode="before")
    @classmethod
    def normalize_query(cls, v):
        if isinstance(v, dict):
            parts = []
            fields = v.get("fields")
            if isinstance(fields, list) and fields:
                parts.append(" ".join(str(item) for item in fields))
            for key in ("query", "query_field", "table", "metric", "line_item", "fiscal_year"):
                value = v.get(key)
                if value is not None and str(value).strip():
                    parts.append(str(value).strip())
            return " | ".join(dict.fromkeys(parts)) if parts else None
        return v


class PlannerOutput(BaseModel):
    needs_decomposition: bool = Field(
        description="True if the question compares multiple time periods, multiple companies, "
                    "or requires data from multiple distinct filings. "
                    "False for single-period, single-company questions."
    )
    sub_queries: List[SubQuery] = Field(
        description="If needs_decomposition=False: exactly one sub-query with the rewritten question. "
                    "If needs_decomposition=True: 2–3 focused sub-queries, one per time period or company."
    )

    @model_validator(mode="before")
    @classmethod
    def normalize_keys(cls, data):
        if not isinstance(data, dict):
            return data
        normalized = dict(data)
        if "sub_queries" not in normalized:
            for alias in ("queries", "sub_query", "query"):
                if alias in normalized:
                    value = normalized[alias]
                    if isinstance(value, list):
                        normalized["sub_queries"] = value
                    else:
                        normalized["sub_queries"] = [value]
                    break
        if "needs_decomposition" not in normalized:
            sub_queries = normalized.get("sub_queries", [])
            normalized["needs_decomposition"] = isinstance(sub_queries, list) and len(sub_queries) > 1
        return normalized

    @field_validator("needs_decomposition", mode="before")
    @classmethod
    def coerce_bool(cls, v): return _coerce_bool(v)


# ── Standard schemas ─────────────────────────────────────────────────────────

class QueryRewriteOutput(BaseModel):
    rewritten_query: str = Field(description="Concise rewritten query optimized for retrieval")


class ContextEvalOutput(BaseModel):
    relevant: bool = Field(description="Whether the retrieved context is relevant and sufficient")
    reason: str = Field(description="Short reason")

    @model_validator(mode="before")
    @classmethod
    def normalize_keys(cls, data):
        if not isinstance(data, dict):
            return data
        normalized = dict(data)
        if "relevant" not in normalized and "is_relevant" in normalized:
            normalized["relevant"] = normalized["is_relevant"]
        answer_text = ""
        if "answer" in normalized:
            answer_text = str(normalized["answer"]).strip().lower()
        reason_text = str(normalized.get("reason", "")).strip().lower()
        combined_text = " ".join(part for part in (answer_text, reason_text) if part)
        if "relevant" not in normalized and "answer" in normalized:
            negative_markers = (
                "not available",
                "n/a",
                "not found",
                "insufficient data",
                "insufficient information",
                "not enough information",
                "cannot determine",
                "unable to determine",
                "cannot answer",
                "does not contain",
                "doesn't contain",
                "cannot infer",
                "unable to infer",
                "missing from the context",
                "not in the context",
            )
            if any(marker in combined_text for marker in negative_markers):
                normalized["relevant"] = False
        if "relevant" not in normalized and "answer" in normalized:
            if reason_text and any(marker in reason_text for marker in ("does not provide", "does not contain", "not provide", "not contain", "cannot determine", "not enough information", "insufficient information")):
                normalized["relevant"] = False
        if "relevant" not in normalized:
            signal_keys = {
                k for k in normalized.keys()
                if k not in {"reason", "is_sufficient", "answer", "is_relevant"}
            }
            if signal_keys:
                normalized["relevant"] = True
        if "reason" not in normalized:
            if "is_sufficient" in normalized:
                normalized["reason"] = f"Model returned is_sufficient={normalized['is_sufficient']}"
            elif "answer" in normalized:
                normalized["reason"] = f"Model returned answer={normalized['answer']}"
            else:
                normalized["reason"] = "No reason provided"
        return normalized

    @field_validator("relevant", mode="before")
    @classmethod
    def coerce_bool(cls, v): return _coerce_bool(v)


# ── Proposal 2: Refined Critic ───────────────────────────────────────────────

class CriticOutput(BaseModel):
    decision: Literal["accept", "repair", "insufficient_data"] = Field(
        description=(
            "accept: the answer is grounded in the context (even if partial). "
            "repair: the answer has a fixable error — wrong period, wrong metric, wrong units, "
            "or directly contradicts the context. The data IS present but the answer is wrong. "
            "insufficient_data: the required financial data is completely absent from ALL retrieved "
            "chunks — not merely incomplete. Never use this just because the question is complex "
            "or the answer is approximate."
        )
    )
    reason: str = Field(description="Short critique")

    @model_validator(mode="before")
    @classmethod
    def normalize_keys(cls, data):
        if not isinstance(data, dict):
            return data
        normalized = dict(data)
        for alias in ("verdict", "label", "result", "status", "answer"):
            if "decision" not in normalized and alias in normalized:
                normalized["decision"] = normalized[alias]
                break
        if "decision" not in normalized:
            if _coerce_bool(normalized.get("repair")) is True:
                normalized["decision"] = "repair"
            elif _coerce_bool(normalized.get("accept")) is True:
                normalized["decision"] = "accept"
            elif _coerce_bool(normalized.get("insufficient_data")) is True:
                normalized["decision"] = "insufficient_data"
            elif _coerce_bool(normalized.get("refuse")) is True:
                normalized["decision"] = "insufficient_data"
        for alias in ("rationale", "explanation", "feedback", "critique", "comment"):
            if "reason" not in normalized and alias in normalized:
                normalized["reason"] = normalized[alias]
                break
        if "reason" not in normalized:
            normalized["reason"] = "No reason provided"
        return normalized

    @field_validator("decision", mode="before")
    @classmethod
    def normalize_decision(cls, v):
        if isinstance(v, str):
            raw = v.strip().lower().replace("-", "_").replace(" ", "_")
            mapping = {
                "accept": "accept",
                "accepted": "accept",
                "approve": "accept",
                "approved": "accept",
                "pass": "accept",
                "grounded": "accept",
                "supported": "accept",
                "repair": "repair",
                "revise": "repair",
                "revised": "repair",
                "fix": "repair",
                "correct": "repair",
                "retry": "repair",
                "insufficient_data": "insufficient_data",
                "insufficient": "insufficient_data",
                "missing_data": "insufficient_data",
                "no_data": "insufficient_data",
                "not_enough_data": "insufficient_data",
                "cannot_answer": "insufficient_data",
                "unanswerable": "insufficient_data",
            }
            return mapping.get(raw, raw)
        return v


class RepairOutput(BaseModel):
    decision: Literal["accept", "warn", "refuse"] = Field(
        description=(
            "Final decision after repair. Use 'accept' when you have a revised answer — "
            "even partial. Use 'warn' only for genuine ambiguity. "
            "Use 'refuse' ONLY if the question is entirely out of scope."
        )
    )
    revised_answer: str = Field(description="The final revised answer.")
    needs_new_retrieval: bool = Field(
        description="true or false — whether fresh retrieval would help. Must be a boolean, not a string."
    )
    reason: str = Field(description="Short rationale")

    @model_validator(mode="before")
    @classmethod
    def normalize_keys(cls, data):
        if not isinstance(data, dict):
            return data
        normalized = dict(data)
        if "data" in normalized and isinstance(normalized["data"], dict):
            for key, value in normalized["data"].items():
                normalized.setdefault(key, value)
        for alias in ("answer", "final_answer", "response"):
            if "revised_answer" not in normalized and alias in normalized:
                normalized["revised_answer"] = normalized[alias]
                break
        if "revised_answer" not in normalized:
            payload_keys = [
                k for k in normalized.keys()
                if k not in {"decision", "reason", "needs_new_retrieval", "data"}
            ]
            if payload_keys:
                normalized["revised_answer"] = "; ".join(
                    f"{k}={normalized[k]}" for k in payload_keys
                )
        for alias in ("rationale", "explanation", "feedback", "comment", "context"):
            if "reason" not in normalized and alias in normalized:
                normalized["reason"] = normalized[alias]
                break
        if "needs_new_retrieval" not in normalized:
            normalized["needs_new_retrieval"] = False
        if "reason" not in normalized:
            if "revised_answer" in normalized:
                normalized["reason"] = "Model returned a revised answer"
            else:
                normalized["reason"] = "No reason provided"
        return normalized
    @field_validator("decision", mode="before")
    @classmethod
    def normalize_decision(cls, v):
        if isinstance(v, str):
            raw = v.strip().lower().replace("-", "_").replace(" ", "_")
            mapping = {
                "accept": "accept",
                "accepted": "accept",
                "warn": "warn",
                "warning": "warn",
                "refuse": "refuse",
                "refusal": "refuse",
                "reject": "refuse",
            }
            return mapping.get(raw, raw)
        return v

    @field_validator("needs_new_retrieval", mode="before")
    @classmethod
    def coerce_bool(cls, v): return _coerce_bool(v)


class JudgeOutput(BaseModel):
    score: float = Field(description="Overall correctness score — a number between 0.0 and 1.0, not a string")
    claims_covered: float = Field(
        description="Fraction of key facts covered — a number between 0.0 and 1.0, not a string"
    )
    reason: str = Field(description="Short explanation")

    @model_validator(mode="before")
    @classmethod
    def normalize_keys(cls, data):
        if not isinstance(data, dict):
            return data
        normalized = dict(data)
        for alias in ("overall_score", "correctness_score", "faithfulness_score", "grounding_score", "rating", "grade"):
            if "score" not in normalized and alias in normalized:
                normalized["score"] = normalized[alias]
                break
        for alias in ("coverage", "coverage_score", "supported_fraction", "support_rate", "claim_coverage"):
            if "claims_covered" not in normalized and alias in normalized:
                normalized["claims_covered"] = normalized[alias]
                break
        for alias in ("rationale", "explanation", "feedback", "comment"):
            if "reason" not in normalized and alias in normalized:
                normalized["reason"] = normalized[alias]
                break
        if isinstance(normalized.get("reason"), dict):
            normalized["reason"] = "; ".join(f"{k}={v}" for k, v in normalized["reason"].items())
        if "claims_covered" not in normalized and "score" in normalized:
            normalized["claims_covered"] = normalized["score"]
        if "reason" not in normalized:
            normalized["reason"] = "No reason provided"
        return normalized

    @field_validator("score", "claims_covered", mode="before")
    @classmethod
    def coerce_float(cls, v): return _coerce_float(v)

In [10]:
# ---------------------------------------------------------------------------
# Prompt templates
# ---------------------------------------------------------------------------

def _question_tokens(text: str) -> set:
    return set(re.findall(r"[A-Za-z0-9]+", str(text).lower()))


def select_few_shot_examples(question: str, max_examples: int = None) -> str:
    if not CONFIG.get("use_few_shot_examples", False) or "train_df" not in globals() or train_df.empty:
        return ""
    top_rows = train_df.sort_values("id").reset_index(drop=True).itertuples(index=False)
    rendered = []
    for idx, row in enumerate(top_rows, start=1):
        rendered.append(
            f"Example {idx}\n"
            f"Question: {getattr(row, 'question', '')}\n"
            f"Reference answer: {getattr(row, 'reference_answer', '')}\n"
            f"Evidence: {getattr(row, 'evidence_text', '')}"
        )
    return "\n\n".join(rendered)


# ── Proposal 1: Planner ───────────────────────────────────────────────────────
planner_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a financial research planner working with SEC filings from these companies:\n"
        "AAPL (Apple), BAC (Bank of America), BRK (Berkshire Hathaway), COST (Costco), "
        "CVX (Chevron), JNJ (Johnson & Johnson), JPM (JPMorgan Chase), MSFT (Microsoft), "
        "NVDA (NVIDIA), UNH (UnitedHealth), WMT (Walmart), XOM (ExxonMobil).\n\n"
        "Decide whether the question requires data from multiple distinct filings, then produce sub-queries.\n\n"
        "Decompose (needs_decomposition=True) when the question:\n"
        "  • Explicitly compares two different fiscal years (e.g. 2023 vs 2024)\n"
        "  • Compares two different companies\n\n"
        "Do NOT decompose single-period, single-company questions.\n\n"
        "For each sub-query set ticker if company-specific, filing_year if year-specific.\n"
        "Every sub-query object must include a non-empty query field.\n"
        "filing_year = the calendar year the 10-K or 10-Q was filed "
        "(e.g. Apple FY2024 10-K was filed in November 2024, so filing_year=2024). "
        "Return a valid JSON object only that matches the requested schema."
    ),
    ("human", "Question: {question}"),
])

context_eval_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You judge whether retrieved context is relevant and sufficient to answer a financial question. "
        "Mark as not relevant only if the context is clearly from the wrong company/period, or completely empty. "
        "Partial tables or incomplete passages still count as relevant if they contain the right metric. "
        "Return a valid JSON object only that matches the requested schema.",
    ),
    ("human", "Question:\n{question}\n\nRetrieved context:\n{context}"),
])

generator_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a financial analyst answering questions using only the retrieved filing context. "
        "Be precise with numbers — include units (millions, billions, %), fiscal periods, and line item names. "
        "If the context does not contain the answer, say so explicitly rather than estimating. "
        "Use the examples only as answer-format/style guidance, never as facts for the current question.",
    ),
    ("human", "Examples:\n{few_shot_examples}\n\nQuestion:\n{question}\n\nRetrieved context:\n{context}"),
])

# ── Proposal 2: Updated critic prompt ────────────────────────────────────────
critic_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a critic for a financial RAG system. Evaluate the answer and choose one decision:\n\n"
        "  accept          — the answer is numerically grounded in the context (even if partial).\n"
        "  repair          — the answer has a fixable error: wrong period, wrong metric, wrong units, "
        "or contradicts the context. The data IS present but the answer used it incorrectly.\n"
        "  insufficient_data — the required financial data is completely absent from ALL retrieved "
        "chunks. Only use this when no amount of repair can help — the filing simply does not "
        "contain the information. Never use this just because the answer is approximate. "
        "Return a valid JSON object only that matches the requested schema.",
    ),
    ("human", "Question:\n{question}\n\nRetrieved context:\n{context}\n\nCurrent answer:\n{answer}"),
])

repair_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You repair a financial RAG answer after critique. Your default action is to ACCEPT with a revised answer. "
        "Pay close attention to: correct fiscal year/quarter, correct units (millions vs billions), "
        "correct sign, and the right line item name. "
        "Set decision='accept' whenever you can produce any useful revised answer — even partial. "
        "Set decision='warn' only if the context is genuinely ambiguous. "
        "Set decision='refuse' ONLY if the question is entirely off-topic for all retrieved filings. "
        "Set needs_new_retrieval=true only if critical data is completely absent from the context. "
        "Return a valid JSON object only that matches the requested schema."
    ),
    (
        "human",
        "Question:\n{question}\n\nRetrieved context:\n{context}\n\n"
        "Original answer:\n{answer}\n\nCritic feedback:\n{critique}",
    ),
])

judge_correctness_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "Score the candidate answer against the reference answer from 0 to 1 for a financial QA task. "
        "1 = correct value, correct units, correct period. "
        "0 = wrong number, wrong company, wrong period, or completely missing. "
        "Give partial credit for answers close but with unit errors. "
        "Also set claims_covered: fraction of distinct facts/numbers in the reference present in the candidate. "
        "Return a valid JSON object only that matches the requested schema.",
    ),
    (
        "human",
        "Question:\n{question}\n\nReference answer:\n{reference_answer}\n\nCandidate answer:\n{candidate_answer}",
    ),
])

judge_faithfulness_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "Score how well the candidate answer is grounded in the retrieved filing context from 0 to 1. "
        "1 = every number and claim is directly supported by the context. "
        "0 = answer contains numbers or claims not present in the context (hallucinated). "
        "Also set claims_covered: fraction of claims in the candidate supported by the context. "
        "Return a valid JSON object only that matches the requested schema.",
    ),
    (
        "human",
        "Question:\n{question}\n\nRetrieved context:\n{context}\n\nCandidate answer:\n{candidate_answer}",
    ),
])

In [11]:
# ---------------------------------------------------------------------------
# Rate limiter + safe LLM invocation helpers
# ---------------------------------------------------------------------------

class RateLimiter:
    def __init__(self, max_rpm: int):
        self.max_rpm = max_rpm
        self.window = 60.0
        self.timestamps: deque = deque()
        self._lock = threading.Lock()

    def wait(self):
        with self._lock:
            now = time.time()
            while self.timestamps and now - self.timestamps[0] >= self.window:
                self.timestamps.popleft()
            if len(self.timestamps) >= self.max_rpm:
                sleep_for = self.window - (now - self.timestamps[0])
                if sleep_for > 0:
                    print(f"  [RateLimit] {self.max_rpm} RPM cap — waiting {sleep_for:.1f}s")
                    time.sleep(sleep_for)
            self.timestamps.append(time.time())


_rate_limiter = RateLimiter(max_rpm=CONFIG["max_rpm"])


def is_rate_limit_error(exc: Exception) -> bool:
    message = str(exc).lower()
    return "rate limit" in message or "rate_limit" in message or "429" in message


def safe_invoke_structured(
    role: str,
    schema_class,
    prompt,
    variables: Dict[str, Any],
    fallback: BaseModel,
    task_name: str = None,
) -> BaseModel:
    # Use json_mode to bypass Groq's strict tool-call schema validation.
    # The model outputs raw JSON which Pydantic then parses — field_validators
    # handle any string→bool/float coercion the model produces.
    model_candidates = order_model_candidates(role, task_name)
    preference_key = task_name or role
    max_attempts = CONFIG["llm_max_retries"]
    for model_idx, (model_name, llm) in enumerate(model_candidates):
        for attempt in range(max_attempts):
            try:
                _rate_limiter.wait()
                structured = llm.with_structured_output(schema_class, method="json_mode")
                result = (prompt | structured).invoke(variables)
                _preferred_models_by_task[preference_key] = model_name
                return result
            except Exception as e:
                print(
                    f"  [WARN] {schema_class.__name__} attempt {attempt+1}/{max_attempts} "
                    f"on {model_name} failed: {e}"
                )
                should_switch = is_rate_limit_error(e) and model_idx < len(model_candidates) - 1
                if should_switch:
                    next_model = model_candidates[model_idx + 1][0]
                    _preferred_models_by_task[preference_key] = next_model
                    print(f"  [WARN] Switching {role} model from {model_name} to {next_model} after rate limit.")
                    break
                if attempt == max_attempts - 1:
                    if model_idx == len(model_candidates) - 1:
                        print(f"  [WARN] All retries exhausted for {schema_class.__name__}, using fallback.")
                        return fallback
                    continue
                delay = CONFIG["llm_retry_base_delay_sec"] * (2 ** attempt)
                time.sleep(delay)
    return fallback


def safe_invoke_llm(
    role: str,
    prompt,
    variables: Dict[str, Any],
    fallback_text: str = "",
    task_name: str = None,
) -> str:
    model_candidates = order_model_candidates(role, task_name)
    preference_key = task_name or role
    max_attempts = CONFIG["llm_max_retries"]
    for model_idx, (model_name, llm) in enumerate(model_candidates):
        chain = prompt | llm
        for attempt in range(max_attempts):
            try:
                _rate_limiter.wait()
                result = chain.invoke(variables).content
                _preferred_models_by_task[preference_key] = model_name
                return result
            except Exception as e:
                print(f"  [WARN] LLM call attempt {attempt+1}/{max_attempts} on {model_name} failed: {e}")
                should_switch = is_rate_limit_error(e) and model_idx < len(model_candidates) - 1
                if should_switch:
                    next_model = model_candidates[model_idx + 1][0]
                    _preferred_models_by_task[preference_key] = next_model
                    print(f"  [WARN] Switching {role} model from {model_name} to {next_model} after rate limit.")
                    break
                if attempt == max_attempts - 1:
                    if model_idx == len(model_candidates) - 1:
                        return fallback_text
                    continue
                delay = CONFIG["llm_retry_base_delay_sec"] * (2 ** attempt)
                time.sleep(delay)
    return fallback_text


def format_retrieved_context(
    chunks: List[RetrievedChunk],
    max_chunks: int = None,
    max_chars: int = None,
) -> str:
    selected = list(chunks[: max_chunks or len(chunks)])
    parts = []
    total_chars = 0
    for i, c in enumerate(selected, start=1):
        parts.append(
            f"[Doc {i}] Company: {c.company} | Filing: {c.doc_name}\n"
            f"Content: {c.raw_chunk}"
        )
        total_chars = sum(len(p) for p in parts) + max(0, 2 * (len(parts) - 1))
        if max_chars is not None and total_chars >= max_chars:
            joined = "\n\n".join(parts)
            return joined[:max_chars]
    return "\n\n".join(parts)


def extract_retrieved_doc_names(chunks: List[RetrievedChunk]) -> List[str]:
    return list(dict.fromkeys([c.doc_name for c in chunks]))


def cleanup_planner_query(query: str, ticker: str = None, filing_year: int = None, form_type: str = None) -> str:
    cleaned = (query or "").strip()
    if not cleaned:
        cleaned = "financial metric from SEC filing"
    upper = cleaned.upper()
    if upper.startswith("SELECT ") and " FROM " in upper:
        cleaned = cleaned.replace("SELECT", "").replace("FROM filings", "").strip()
    for token in ("WHERE", "FROM", "AND", "=", "'", '"'):
        cleaned = cleaned.replace(token, " ")
    cleaned = cleaned.replace("filing year", "filing_year")
    cleaned = cleaned.replace("|", " ")
    cleaned = cleaned.replace("_", " ")
    cleaned = cleaned.replace("company", "")
    cleaned = cleaned.replace("ticker", "")
    cleaned = " ".join(cleaned.split())
    parts = [cleaned]
    if ticker and ticker not in cleaned:
        parts.insert(0, ticker)
    if filing_year and str(filing_year) not in cleaned:
        parts.append(str(filing_year))
    if form_type and form_type not in cleaned:
        parts.append(form_type)
    return " ".join(part for part in parts if part).strip()


def fails_retrieval_sanity_check(question: str, sub_queries: List[Dict[str, Any]], retrieved: List[RetrievedChunk]) -> bool:
    if not CONFIG.get("enable_retrieval_sanity_check", False):
        return False
    if not retrieved:
        return False
    expected_tickers = {sq.get("ticker") for sq in sub_queries if sq.get("ticker")}
    if not expected_tickers:
        return False
    present_tickers = {getattr(chunk, "ticker", None) for chunk in retrieved if getattr(chunk, "ticker", None)}
    if not present_tickers:
        return False
    return not expected_tickers.issubset(present_tickers)


def get_active_model_name(role: str, task_name: str = None) -> str:
    candidates = order_model_candidates(role, task_name)
    return candidates[0][0] if candidates else resolve_model_name(role)


def get_model_aware_context_limits(
    role: str,
    task_name: str,
    default_chunks: int,
    default_chars: int,
) -> Tuple[int, int]:
    model_name = get_active_model_name(role, task_name).lower()
    if "llama-3.1-8b-instant" in model_name:
        budgets = {
            "context_eval": (2, 2200),
            "critic": (2, 2500),
            "repair": (2, 2500),
            "judge": (2, 2200),
            "generator": (5, 7000),
        }
        return budgets.get(task_name, (default_chunks, default_chars))
    if "qwen/qwen3-32b" in model_name:
        budgets = {
            "context_eval": (3, 3500),
            "critic": (3, 4000),
            "repair": (3, 4000),
            "judge": (3, 3200),
            "generator": (6, 9000),
        }
        return budgets.get(task_name, (default_chunks, default_chars))
    return default_chunks, default_chars

In [12]:

# ─────────────────────────────────────────────────────────────────────────────
# DRIFT DETECTION + AUTO-INGESTION INFRASTRUCTURE
# ─────────────────────────────────────────────────────────────────────────────
# Detects when users consistently query for data not yet in the corpus
# (e.g. FY2025 filings when the DB only has up to FY2024), then automatically
# scrapes the missing filing from SEC EDGAR and ingests it into ChromaDB.
#
# Graph flow (new branch from critic's insufficient_data path):
#   critic (insufficient_data)
#     → node_drift_detector  [logs miss, checks TRIGGER_THRESHOLD]
#       → node_scrape_and_ingest  [EDGAR → chunk → upsert ChromaDB → rebuild BM25]
#         → hybrid_retriever  [retry with freshly ingested data]
# ─────────────────────────────────────────────────────────────────────────────

import requests
from collections import defaultdict
from datetime import datetime
from pathlib import Path

# ── Ticker → CIK mapping (same 12 companies as the corpus) ──────────────────
TICKER_TO_CIK = {
    "AAPL":  "0000320193", "MSFT":  "0000789019", "NVDA":  "0001045810",
    "JPM":   "0000019617", "BAC":   "0000070858", "BRK-B": "0001067983",
    "JNJ":   "0000200406", "UNH":   "0000731766", "XOM":   "0000034088",
    "CVX":   "0000093410", "WMT":   "0000104169", "COST":  "0000909832",
}

COMPANY_NAMES_MAP = {
    "AAPL":  "Apple Inc.",                         "MSFT":  "Microsoft Corporation",
    "NVDA":  "NVIDIA Corporation",                 "JPM":   "JPMorgan Chase & Co.",
    "BAC":   "Bank of America Corporation",        "BRK-B": "Berkshire Hathaway Inc.",
    "JNJ":   "Johnson & Johnson",                  "UNH":   "UnitedHealth Group Incorporated",
    "XOM":   "Exxon Mobil Corporation",            "CVX":   "Chevron Corporation",
    "WMT":   "Walmart Inc.",                       "COST":  "Costco Wholesale Corporation",
}

# EDGAR requires a descriptive User-Agent with contact info
EDGAR_HEADERS = {"User-Agent": "GenAI-RAG-Project research@example.com"}


# ─────────────────────────────────────────────────────────────────────────────
# DriftTracker — accumulates corpus-miss signals and decides when to scrape
# ─────────────────────────────────────────────────────────────────────────────

class DriftTracker:
    """
    Logs each 'insufficient_data' critic decision and counts misses per
    (ticker, filing_year, form_type). Once a combo hits TRIGGER_THRESHOLD,
    it schedules a scrape from EDGAR on the next qualifying graph run.
    The log is persisted to JSONL so counts survive session restarts.
    """
    TRIGGER_THRESHOLD = 2
    PERSIST_PATH = Path(CONFIG["sec_chunks_path"]).parent.parent / "drift_log.jsonl"

    def __init__(self):
        self._counts: Dict[tuple, int] = defaultdict(int)
        self._already_ingested: set = set()
        self._load_existing()

    def _load_existing(self):
        if self.PERSIST_PATH.exists():
            with open(self.PERSIST_PATH, encoding="utf-8") as f:
                for line in f:
                    try:
                        entry = json.loads(line)
                        key = (entry.get("ticker"), entry.get("filing_year"), entry.get("form_type"))
                        if entry.get("status") == "ingested":
                            self._already_ingested.add(key)
                        elif entry.get("status") == "miss":
                            self._counts[key] += 1
                    except Exception:
                        pass

    def log_miss(self, question: str, ticker: Optional[str],
                 filing_year: Optional[str], form_type: Optional[str]) -> None:
        key = (ticker, filing_year, form_type or "10-K")
        if key in self._already_ingested:
            return
        self._counts[key] += 1
        self.PERSIST_PATH.parent.mkdir(parents=True, exist_ok=True)
        with open(self.PERSIST_PATH, "a", encoding="utf-8") as f:
            f.write(json.dumps({
                "status": "miss", "ticker": ticker, "filing_year": filing_year,
                "form_type": form_type or "10-K", "question": question,
                "timestamp": datetime.utcnow().isoformat(),
            }) + "\n")
        print(f"  [DriftTracker] Miss #{self._counts[key]} for ({ticker}, {filing_year}, {form_type or '10-K'})")

    def get_pending_scrapes(self) -> List[Tuple[str, str, str]]:
        """Return (ticker, year, form_type) combos that crossed the threshold and are not yet ingested."""
        return [
            (ticker, year, ft)
            for (ticker, year, ft), count in self._counts.items()
            if count >= self.TRIGGER_THRESHOLD
            and (ticker, year, ft) not in self._already_ingested
            and ticker and year
        ]

    def mark_ingested(self, ticker: str, filing_year: str, form_type: str) -> None:
        key = (ticker, filing_year, form_type)
        self._already_ingested.add(key)
        with open(self.PERSIST_PATH, "a", encoding="utf-8") as f:
            f.write(json.dumps({
                "status": "ingested", "ticker": ticker,
                "filing_year": filing_year, "form_type": form_type,
                "timestamp": datetime.utcnow().isoformat(),
            }) + "\n")


drift_tracker = DriftTracker()
print(f"DriftTracker ready | persist path: {drift_tracker.PERSIST_PATH}")


# ─────────────────────────────────────────────────────────────────────────────
# SEC EDGAR Scraper
# ─────────────────────────────────────────────────────────────────────────────

def _strip_html(html: str) -> str:
    text = re.sub(r"<[^>]+>", " ", html)
    for ent, repl in [("&nbsp;", " "), ("&amp;", "&"), ("&lt;", "<"), ("&gt;", ">"), ("&#160;", " ")]:
        text = text.replace(ent, repl)
    return re.sub(r"\s{3,}", "\n\n", text).strip()


def fetch_edgar_filings_list(ticker: str, form_type: str = "10-K") -> List[Dict]:
    """Returns recent filings metadata from EDGAR submissions API for a given ticker and form type."""
    cik = TICKER_TO_CIK.get(ticker.upper())
    if not cik:
        raise ValueError(f"Unknown ticker: {ticker}. Add it to TICKER_TO_CIK.")
    resp = requests.get(
        f"https://data.sec.gov/submissions/CIK{cik}.json",
        headers=EDGAR_HEADERS, timeout=30,
    )
    resp.raise_for_status()
    filings = resp.json().get("filings", {}).get("recent", {})
    return [
        {
            "accessionNumber": filings["accessionNumber"][i],
            "filingDate":      filings["filingDate"][i],
            "form":            form,
            "primaryDocument": (filings.get("primaryDocument") or [None])[i],
        }
        for i, form in enumerate(filings.get("form", []))
        if form == form_type
    ]


def fetch_filing_text(ticker: str, accession_number: str, primary_doc: Optional[str]) -> str:
    """Downloads and strips HTML from the primary document of a given accession number."""
    cik = TICKER_TO_CIK[ticker.upper()].lstrip("0")
    acc_no_dashes = accession_number.replace("-", "")
    base = f"https://www.sec.gov/Archives/edgar/data/{cik}/{acc_no_dashes}"
    if primary_doc:
        resp = requests.get(f"{base}/{primary_doc}", headers=EDGAR_HEADERS, timeout=60)
        if resp.ok:
            return _strip_html(resp.text)
    # Fallback: parse the filing index page for the largest .htm document
    index_url = f"{base}/{accession_number}-index.htm"
    resp = requests.get(index_url, headers=EDGAR_HEADERS, timeout=30)
    links = re.findall(r'href="(/Archives[^"]+\.htm)"', resp.text, re.IGNORECASE)
    if links:
        resp2 = requests.get(f"https://www.sec.gov{links[0]}", headers=EDGAR_HEADERS, timeout=60)
        return _strip_html(resp2.text)
    raise RuntimeError(f"Could not fetch filing for {ticker} {accession_number}")


def scrape_filing(ticker: str, filing_year: str, form_type: str = "10-K") -> Dict:
    """
    Finds and downloads the primary SEC filing document for a given ticker/year/form from EDGAR.
    Tries the exact calendar year first, then year+1 (common for 10-Ks filed in Jan/Feb).
    """
    filings = fetch_edgar_filings_list(ticker, form_type)
    match = next((f for f in filings if f["filingDate"].startswith(str(filing_year))), None)
    if match is None:
        match = next((f for f in filings if f["filingDate"].startswith(str(int(filing_year) + 1))), None)
    if match is None:
        raise RuntimeError(f"No {form_type} found for {ticker} FY{filing_year} on EDGAR.")
    text = fetch_filing_text(ticker, match["accessionNumber"], match.get("primaryDocument"))
    print(f"  [EDGAR] Downloaded {ticker} {form_type} {filing_year} "
          f"({len(text):,} chars, filed {match['filingDate']})")
    return {
        "ticker":           ticker,
        "filing_year":      filing_year,
        "form_type":        form_type,
        "filing_date":      match["filingDate"],
        "accession_number": match["accessionNumber"],
        "text":             text,
    }


# ─────────────────────────────────────────────────────────────────────────────
# Chunker + ChromaDB Ingestor
# ─────────────────────────────────────────────────────────────────────────────

DRIFT_CHUNK_WORDS   = 800
DRIFT_CHUNK_OVERLAP = 100


def chunk_filing(filing: Dict) -> List[Dict]:
    """Splits a raw filing text into overlapping word-window chunks matching sec_chunks.jsonl schema."""
    words   = filing["text"].split()
    company = COMPANY_NAMES_MAP.get(filing["ticker"], filing["ticker"])
    chunks, start, idx = [], 0, 0
    while start < len(words):
        end  = min(start + DRIFT_CHUNK_WORDS, len(words))
        text = " ".join(words[start:end])
        cid  = f"{filing['ticker']}_{filing['form_type']}_{filing['filing_year']}_{idx:04d}_drift"
        contextual = (
            f"Company: {company} ({filing['ticker']})\n"
            f"Filing: {filing['form_type']} | Date: {filing['filing_date']} | Year: {filing['filing_year']}\n"
            f"Section: auto-chunked\n"
            f"Content: {text}"
        )
        chunks.append({
            "chunk_id":                cid,
            "ticker":                  filing["ticker"],
            "company_name":            company,
            "form_type":               filing["form_type"],
            "filing_year":             str(filing["filing_year"]),
            "filing_date":             filing["filing_date"],
            "accession_number":        filing["accession_number"],
            "section_title":           "auto-chunked",
            "chunk_index":             idx,
            "text":                    text,
            "contextual_text":         contextual,
            "word_count":              end - start,
            "source":                  "drift_ingested",
            "flag_low_value_combined": False,
        })
        start += DRIFT_CHUNK_WORDS - DRIFT_CHUNK_OVERLAP
        idx   += 1
    print(f"  [Chunker] {len(chunks)} chunks from {filing['ticker']} {filing['form_type']} {filing['filing_year']}")
    return chunks


def ingest_to_chroma(chunks: List[Dict], corpus_index: CorpusIndex) -> int:
    """
    Embeds and upserts new chunks into the CorpusIndex's existing ChromaDB collection.
    Skips chunks whose IDs are already present (idempotent).
    """
    collection = corpus_index.chroma_col
    existing   = set(collection.get(ids=[c["chunk_id"] for c in chunks])["ids"])
    new        = [c for c in chunks if c["chunk_id"] not in existing]
    if not new:
        print("  [Ingestor] All chunks already present — skipping.")
        return 0
    texts = [c["contextual_text"] for c in new]
    embs  = embed_model.encode(texts, batch_size=32, show_progress_bar=False).tolist()
    metas = [{k: v for k, v in c.items() if k not in ("text", "contextual_text")} for c in new]
    collection.upsert(
        ids=       [c["chunk_id"] for c in new],
        embeddings=embs,
        documents= texts,
        metadatas= metas,
    )
    print(f"  [Ingestor] Added {len(new)} chunks to ChromaDB.")
    return len(new)


def update_corpus_index(corpus_index: CorpusIndex, new_chunks: List[Dict]) -> None:
    """
    Hot-patches the live CorpusIndex with freshly scraped chunks.
    Appends rows to the df, rebuilds BM25, and updates the str→row lookup —
    all in-memory so the current session benefits without a kernel restart.
    Also appends new chunks to sec_chunks.jsonl so future sessions load them automatically.
    """
    base_len = len(corpus_index.df)
    new_rows = []
    for i, c in enumerate(new_chunks):
        doc_name   = f"{c['ticker']}_{c['form_type']}_{c['filing_date'][:10]}"
        contextual = c["contextual_text"]
        new_rows.append({
            "chunk_id":         base_len + i,
            "chunk_id_str":     c["chunk_id"],
            "ticker":           c["ticker"],
            "company":          c["company_name"],
            "doc_name":         doc_name,
            "filing_year":      int(c["filing_year"]),
            "form_type":        c["form_type"],
            "page_num":         c["chunk_index"],
            "raw_chunk":        c["text"],
            "contextual_chunk": contextual,
            "bm25_tokens":      contextual.lower().split(),
        })
    new_df = pd.DataFrame(new_rows)
    corpus_index.df   = pd.concat([corpus_index.df, new_df], ignore_index=True)
    corpus_index.bm25 = BM25Okapi(corpus_index.df["bm25_tokens"].tolist())
    # Update the str→row lookup used by dense_search for BM25↔ChromaDB cross-referencing
    for i, c in enumerate(new_chunks):
        corpus_index._str_to_row[c["chunk_id"]] = base_len + i
    # Persist to JSONL so future sessions load new data without re-scraping
    with open(CONFIG["sec_chunks_path"], "a", encoding="utf-8") as f:
        for c in new_chunks:
            f.write(json.dumps(c) + "\n")
    print(f"  [BM25] Index rebuilt: {len(corpus_index.df):,} total chunks.")


DriftTracker ready | persist path: C:\Users\jeeey\GenAI-with-LLMs\sec_rag_team_share\sec_data\drift_log.jsonl


In [13]:
# ---------------------------------------------------------------------------
# LangGraph state + nodes
# ---------------------------------------------------------------------------

class GraphState(TypedDict, total=False):
    question:              str
    index:                 CorpusIndex
    # Planner outputs
    rewritten_query:       str
    sub_queries:           List[Dict]   # list of SubQuery dicts
    needs_decomposition:   bool
    # Retrieval
    retrieved_chunks:      List[RetrievedChunk]
    retrieved_doc_names:   List[str]
    retrieval_sanity_failed: bool
    # Context eval
    context_retries:       int
    context_eval_relevant: bool
    # Generation
    generated_answer:      str
    # Critic
    critic_decision:       str
    critic_reason:         str
    # Repair
    repair_used:           bool
    repair_decision:       str
    repair_reason:         str
    needs_new_retrieval:   bool
    repair_retrieval_count: int
    is_repair_retrieval:   bool
    # Final
    final_answer:          str
    # Drift detection
    drift_pending_scrapes: List[Tuple]  # [(ticker, filing_year, form_type), ...]
    drift_triggered:       bool
    ingestion_just_ran:    bool


# ── Proposal 1: Planner node ─────────────────────────────────────────────────

def node_query_planner(state: GraphState) -> GraphState:
    result = safe_invoke_structured(
        "agent",
        PlannerOutput,
        planner_prompt,
        {"question": state["question"]},
        PlannerOutput(
            needs_decomposition=False,
            sub_queries=[SubQuery(query=state["question"])],
        ),
        task_name="planner",
    )
    sub_queries = []
    for sq in result.sub_queries:
        sq_dict = sq.model_dump()
        query = (sq_dict.get("query") or "").strip()
        if not query:
            parts = [state["question"]]
            if sq_dict.get("ticker"):
                parts.append(f"ticker {sq_dict['ticker']}")
            if sq_dict.get("filing_year"):
                parts.append(f"filing year {sq_dict['filing_year']}")
            if sq_dict.get("form_type"):
                parts.append(f"form {sq_dict['form_type']}")
            query = " | ".join(parts)
        sq_dict["query"] = cleanup_planner_query(
            query,
            ticker=sq_dict.get("ticker"),
            filing_year=sq_dict.get("filing_year"),
            form_type=sq_dict.get("form_type"),
        )
        sub_queries.append(sq_dict)
    rewritten_query = sub_queries[0]["query"] if sub_queries else state["question"]
    n = len(sub_queries)
    label = "decomposed" if result.needs_decomposition else "single"
    print(f"  [Planner] {label} | {n} sub-quer{'ies' if n > 1 else 'y'}")
    if result.needs_decomposition:
        for sq in sub_queries:
            print(f"    → {sq['query']}  (ticker={sq.get('ticker')}, year={sq.get('filing_year')})")
    return {
        "rewritten_query":     rewritten_query,
        "sub_queries":         sub_queries,
        "needs_decomposition": result.needs_decomposition,
    }


# ── Updated hybrid retriever (multi-query aware) ─────────────────────────────

def node_hybrid_retriever(state: GraphState) -> GraphState:
    sub_queries = state.get("sub_queries", [])

    if len(sub_queries) <= 1:
        # Single-query path
        query = state.get("rewritten_query", state["question"])
        sq = sub_queries[0] if sub_queries else {}
        retrieved = state["index"].hybrid_search(
            query,
            bm25_top_k=CONFIG["bm25_top_k"],
            dense_top_k=CONFIG["dense_top_k"],
            rerank_top_k=CONFIG["rerank_top_k"],
            ticker=sq.get("ticker"),
            filing_year=sq.get("filing_year"),
            form_type=sq.get("form_type"),
        )
    else:
        # Multi-query decomposition: retrieve per sub-query then merge by best score
        per_k = CONFIG["decomposition_top_k_per_subquery"]
        merged: Dict[int, RetrievedChunk] = {}
        for sq in sub_queries:
            chunks = state["index"].hybrid_search(
                sq["query"],
                bm25_top_k=CONFIG["bm25_top_k"],
                dense_top_k=CONFIG["dense_top_k"],
                rerank_top_k=per_k,
                ticker=sq.get("ticker"),
                filing_year=sq.get("filing_year"),
                form_type=sq.get("form_type"),
            )
            for chunk in chunks:
                key = chunk.chunk_id
                if key not in merged or chunk.score > merged[key].score:
                    merged[key] = chunk
        retrieved = sorted(merged.values(), key=lambda x: x.score, reverse=True)[:CONFIG["rerank_top_k"]]

    sanity_failed = fails_retrieval_sanity_check(state["question"], sub_queries, retrieved)
    return {
        "retrieved_chunks":         retrieved,
        "retrieved_doc_names":      extract_retrieved_doc_names(retrieved),
        "retrieval_sanity_failed":  sanity_failed,
    }


def node_context_evaluator(state: GraphState) -> GraphState:
    if state.get("retrieval_sanity_failed", False):
        return {"context_eval_relevant": False, "context_retries": state.get("context_retries", 0)}
    max_chunks, max_chars = get_model_aware_context_limits(
        "agent", "context_eval", CONFIG["control_max_context_chunks"], CONFIG["control_max_context_chars"]
    )
    context = format_retrieved_context(
        state.get("retrieved_chunks", []),
        max_chunks=max_chunks,
        max_chars=max_chars,
    )
    result = safe_invoke_structured(
        "agent",
        ContextEvalOutput,
        context_eval_prompt,
        {"question": state["question"], "context": context},
        ContextEvalOutput(relevant=True, reason="Fallback accept"),
        task_name="context_eval",
    )
    return {"context_eval_relevant": result.relevant, "context_retries": state.get("context_retries", 0)}


def route_context(state: GraphState) -> str:
    if state.get("context_eval_relevant", True):
        return "generator"
    if state.get("context_retries", 0) < CONFIG["max_context_retries"]:
        return "retry_retrieval"
    return "generator"


def node_increment_context_retry(state: GraphState) -> GraphState:
    return {"context_retries": state.get("context_retries", 0) + 1}


def node_generator(state: GraphState) -> GraphState:
    max_chunks, max_chars = get_model_aware_context_limits(
        "generator", "generator", CONFIG["generator_max_context_chunks"], CONFIG["generator_max_context_chars"]
    )
    few_shot_examples = select_few_shot_examples(state["question"])
    context = format_retrieved_context(
        state.get("retrieved_chunks", []),
        max_chunks=max_chunks,
        max_chars=max_chars,
    )
    raw = safe_invoke_llm(
        "generator",
        generator_prompt,
        {"question": state["question"], "context": context, "few_shot_examples": few_shot_examples},
        task_name="generator",
    )
    answer = normalize_answer_text(raw)
    return {"generated_answer": answer, "final_answer": answer}


# ── Proposal 2: Updated critic node + routing ─────────────────────────────────

def node_critic(state: GraphState) -> GraphState:
    max_chunks, max_chars = get_model_aware_context_limits(
        "agent", "critic", CONFIG["control_max_context_chunks"], CONFIG["control_max_context_chars"]
    )
    context = format_retrieved_context(
        state.get("retrieved_chunks", []),
        max_chunks=max_chunks,
        max_chars=max_chars,
    )
    result = safe_invoke_structured(
        "agent",
        CriticOutput,
        critic_prompt,
        {
            "question": state["question"],
            "context":  context,
            "answer":   state.get("generated_answer", ""),
        },
        CriticOutput(decision="accept", reason="Fallback accept"),
        task_name="critic",
    )
    updates: GraphState = {"critic_decision": result.decision, "critic_reason": result.reason}
    # Proposal 2: if data is absent, set the final answer immediately so we exit cleanly
    if result.decision == "insufficient_data":
        updates["final_answer"] = (
            f"Insufficient data: The required information could not be found in the "
            f"retrieved SEC filings. ({result.reason})"
        )
    return updates


def route_critic(state: GraphState) -> str:
    if state.get("is_repair_retrieval", False):
        return "end"
    decision = state.get("critic_decision", "accept")
    if decision == "repair":
        return "repair"
    if decision == "insufficient_data":
        return "insufficient_data"
    return "end"


def node_repair(state: GraphState) -> GraphState:
    max_chunks, max_chars = get_model_aware_context_limits(
        "agent", "repair", CONFIG["control_max_context_chunks"], CONFIG["control_max_context_chars"]
    )
    context = format_retrieved_context(
        state.get("retrieved_chunks", []),
        max_chunks=max_chunks,
        max_chars=max_chars,
    )
    result = safe_invoke_structured(
        "agent",
        RepairOutput,
        repair_prompt,
        {
            "question": state["question"],
            "context":  context,
            "answer":   state.get("generated_answer", ""),
            "critique": state.get("critic_reason", "Needs repair"),
        },
        RepairOutput(
            decision="accept",
            revised_answer=state.get("generated_answer", ""),
            needs_new_retrieval=False,
            reason="Fallback repair",
        ),
        task_name="repair",
    )
    count = state.get("repair_retrieval_count", 0)
    if result.needs_new_retrieval:
        count += 1
    return {
        "repair_used":           True,
        "repair_decision":       result.decision,
        "repair_reason":         result.reason,
        "needs_new_retrieval":   result.needs_new_retrieval,
        "repair_retrieval_count": count,
        "final_answer":          normalize_answer_text(result.revised_answer),
    }


def route_repair(state: GraphState) -> str:
    if state.get("needs_new_retrieval", False) and state.get("repair_retrieval_count", 0) <= CONFIG["max_repair_retrievals"]:
        return "re_retrieve"
    return "end"


def node_mark_repair_retrieval(state: GraphState) -> GraphState:
    return {"is_repair_retrieval": True}

# ── Drift detection nodes (Proposal 3) ───────────────────────────────────────

def node_drift_detector(state: GraphState) -> GraphState:
    """
    Called after the critic returns 'insufficient_data'.
    Extracts the ticker/year/form from the planner's sub-queries, logs the miss
    to DriftTracker, then checks whether the scrape threshold has been crossed.
    """
    sub_queries = state.get("sub_queries", [])
    ticker, filing_year, form_type = None, None, None
    if sub_queries:
        sq         = sub_queries[0]
        ticker     = sq.get("ticker")
        filing_year = str(sq.get("filing_year")) if sq.get("filing_year") else None
        form_type  = sq.get("form_type") or "10-K"

    drift_tracker.log_miss(
        question    = state["question"],
        ticker      = ticker,
        filing_year = filing_year,
        form_type   = form_type,
    )

    pending = drift_tracker.get_pending_scrapes()
    print(f"  [DriftDetector] Pending scrapes: {pending}")

    return {
        "drift_pending_scrapes": pending,
        "drift_triggered":       len(pending) > 0,
        "ingestion_just_ran":    False,
    }


def route_drift(state: GraphState) -> str:
    """Scrape if threshold crossed and data is available on EDGAR; otherwise exit cleanly."""
    if state.get("drift_triggered") and state.get("drift_pending_scrapes"):
        return "scrape_and_ingest"
    return "end"


def node_scrape_and_ingest(state: GraphState) -> GraphState:
    """
    Scrapes each pending (ticker, year, form_type) from EDGAR, chunks the text,
    upserts into ChromaDB, and hot-patches the live CorpusIndex's BM25 index.
    Clears critic state so the graph retries retrieval with the fresh data.
    """
    corpus_index = state["index"]
    ingested_any = False

    for ticker, filing_year, form_type in state.get("drift_pending_scrapes", []):
        try:
            filing = scrape_filing(ticker, str(filing_year), form_type)
            chunks = chunk_filing(filing)
            added  = ingest_to_chroma(chunks, corpus_index)
            if added > 0:
                update_corpus_index(corpus_index, chunks)
                ingested_any = True
            drift_tracker.mark_ingested(ticker, str(filing_year), form_type)
        except Exception as e:
            print(f"  [ScrapeIngest] Failed for {ticker} {filing_year}: {e}")

    return {
        "ingestion_just_ran":  ingested_any,
        "context_retries":     0,
        "critic_decision":     None,
        "is_repair_retrieval": False,
    }


def route_after_scrape(state: GraphState) -> str:
    """If new data was ingested, route back to retrieval for a second attempt; otherwise exit."""
    if state.get("ingestion_just_ran"):
        return "retry_retrieval"
    return "end"


In [14]:
# ---------------------------------------------------------------------------
# Graph construction
# Entry point changed: query_planner replaces query_rewriter
# ---------------------------------------------------------------------------

def build_agentic_graph():
    g = StateGraph(GraphState)

    g.add_node("query_planner",          node_query_planner)
    g.add_node("hybrid_retriever",        node_hybrid_retriever)
    g.add_node("context_evaluator",       node_context_evaluator)
    g.add_node("increment_context_retry", node_increment_context_retry)
    g.add_node("generator",               node_generator)
    g.add_node("critic",                  node_critic)
    g.add_node("repair",                  node_repair)
    g.add_node("mark_repair_retrieval",   node_mark_repair_retrieval)
    g.add_node("drift_detector",          node_drift_detector)
    g.add_node("scrape_and_ingest",       node_scrape_and_ingest)

    g.set_entry_point("query_planner")
    g.add_edge("query_planner",          "hybrid_retriever")
    g.add_edge("hybrid_retriever",        "context_evaluator")
    g.add_conditional_edges(
        "context_evaluator",
        route_context,
        {"generator": "generator", "retry_retrieval": "increment_context_retry"},
    )
    g.add_edge("increment_context_retry", "hybrid_retriever")
    g.add_edge("generator",               "critic")
    g.add_conditional_edges(
        "critic",
        route_critic,
        {"repair": "repair", "end": END, "insufficient_data": "drift_detector"},
    )
    g.add_conditional_edges(
        "drift_detector",
        route_drift,
        {"scrape_and_ingest": "scrape_and_ingest", "end": END},
    )
    g.add_conditional_edges(
        "scrape_and_ingest",
        route_after_scrape,
        {"retry_retrieval": "hybrid_retriever", "end": END},
    )
    g.add_conditional_edges(
        "repair",
        route_repair,
        {"re_retrieve": "mark_repair_retrieval", "end": END},
    )
    g.add_edge("mark_repair_retrieval",   "hybrid_retriever")

    return g.compile()


agentic_graph = build_agentic_graph()
print("Agentic graph compiled (with drift detection).")

Agentic graph compiled (with drift detection).


In [15]:
# ---------------------------------------------------------------------------
# Metrics helpers
# ---------------------------------------------------------------------------

def normalize_answer_text(answer: Any) -> str:
    if answer is None:
        return ""
    if isinstance(answer, str):
        return answer.strip()
    if isinstance(answer, list):
        return " ".join(str(x) for x in answer if x is not None).strip()
    if isinstance(answer, dict):
        for key in ["final_answer", "answer", "text", "content", "generated_answer"]:
            if key in answer:
                return normalize_answer_text(answer[key])
        return str(answer).strip()
    return str(answer).strip()


def normalize_for_eval(text: Any) -> str:
    text = normalize_answer_text(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def token_f1_score(prediction: Any, reference: Any) -> float:
    pred_tokens = normalize_for_eval(prediction).split()
    ref_tokens  = normalize_for_eval(reference).split()
    if not pred_tokens and not ref_tokens:
        return 1.0
    if not pred_tokens or not ref_tokens:
        return 0.0
    common = Counter(pred_tokens) & Counter(ref_tokens)
    n_same = sum(common.values())
    if n_same == 0:
        return 0.0
    p = n_same / len(pred_tokens)
    r = n_same / len(ref_tokens)
    return 2 * p * r / (p + r)


def token_precision_recall(prediction: Any, reference: Any) -> Tuple[float, float]:
    pred_tokens = normalize_for_eval(prediction).split()
    ref_tokens  = normalize_for_eval(reference).split()
    if not pred_tokens and not ref_tokens:
        return 1.0, 1.0
    if not pred_tokens or not ref_tokens:
        return 0.0, 0.0
    common = Counter(pred_tokens) & Counter(ref_tokens)
    n_same = sum(common.values())
    if n_same == 0:
        return 0.0, 0.0
    return n_same / len(pred_tokens), n_same / len(ref_tokens)


def _extract_numbers(text: str) -> List[float]:
    text = text.lower()
    for word, mult in [("trillion", 1e12), ("billion", 1e9), ("million", 1e6), ("thousand", 1e3)]:
        text = re.sub(
            rf"([\d,.]+)\s*{word}",
            lambda m, mult=mult: str(float(m.group(1).replace(",", "")) * mult),
            text,
        )
    numbers = []
    for tok in re.findall(r"\(?\$?[\d,]+\.?\d*\)?", text):
        negative = tok.startswith("(") and tok.endswith(")")
        try:
            val = float(re.sub(r"[^\d.]", "", tok))
            numbers.append(-val if negative else val)
        except ValueError:
            pass
    return numbers


def numerical_accuracy_score(prediction: Any, reference: Any, tolerance: float = 0.01) -> float:
    pred_nums = _extract_numbers(normalize_answer_text(prediction))
    ref_nums  = _extract_numbers(normalize_answer_text(reference))
    if not ref_nums:
        return 1.0
    if not pred_nums:
        return 0.0
    for ref_val in ref_nums:
        for pred_val in pred_nums:
            if ref_val == 0 and pred_val == 0:
                return 1.0
            if ref_val != 0 and abs(pred_val - ref_val) / abs(ref_val) <= tolerance:
                return 1.0
    return 0.0


def compute_decision_from_text(answer: Any) -> str:
    lower = normalize_answer_text(answer).lower()
    if any(x in lower for x in ["insufficient data", "cannot answer", "not available", "not found", "do not know"]):
        return "warn"
    if any(x in lower for x in ["refuse", "cannot comply"]):
        return "refuse"
    return "answer"


def retrieval_metrics(
    retrieved_chunks: Any,
    evidence_chunk_id: str = "",
    ticker: str = "",
    filing_year: Any = None,
    form_type: str = "",
) -> Tuple[float, float, float]:
    chunks = retrieved_chunks if isinstance(retrieved_chunks, list) else []
    if not chunks:
        return 0.0, 0.0, 0.0
    retrieved_ids = [getattr(chunk, 'chunk_id', None) for chunk in chunks]
    recall = 1.0 if evidence_chunk_id and evidence_chunk_id in retrieved_ids else 0.0
    precision = (1.0 / len(chunks)) if recall > 0 else 0.0
    relevant_count = 0
    target_ticker = str(ticker or '').strip().upper()
    target_form = str(form_type or '').strip().upper()
    target_year = int(filing_year) if filing_year not in (None, '', float('nan')) and pd.notna(filing_year) else None
    for chunk in chunks:
        same_ticker = not target_ticker or str(getattr(chunk, 'ticker', '')).strip().upper() == target_ticker
        same_form = not target_form or str(getattr(chunk, 'form_type', '')).strip().upper() == target_form
        same_year = target_year is None or getattr(chunk, 'filing_year', None) == target_year
        if same_ticker and same_form and same_year:
            relevant_count += 1
    relevancy = relevant_count / len(chunks) if chunks else 0.0
    return precision, recall, relevancy


def correct_answer_binary(
    expected_decision: str,
    llm_correctness_score: Any,
    numerical_accuracy: Any,
    token_f1: Any,
    task_completion: float,
) -> int:
    expected_decision = str(expected_decision or '').strip().lower()
    if expected_decision != 'answer':
        return int(task_completion >= 1.0)
    if llm_correctness_score is not None and pd.notna(llm_correctness_score):
        return int(float(llm_correctness_score) >= 0.8)
    if numerical_accuracy is not None and pd.notna(numerical_accuracy) and float(numerical_accuracy) >= 1.0:
        return 1
    if token_f1 is not None and pd.notna(token_f1) and float(token_f1) >= 0.8:
        return 1
    return 0


def build_agent_route_trace(state: Dict[str, Any]) -> str:
    steps = ["planner"]
    if state.get("needs_decomposition"):
        steps.append("decompose")
    steps.append("retrieve")
    if state.get("retrieval_sanity_failed"):
        steps.append("retrieval_sanity_failed")
    if (state.get("context_retries") or 0) > 0:
        steps.append(f"context_retry={state.get('context_retries', 0)}")
    steps.append("generate")
    critic_decision = state.get("critic_decision")
    if critic_decision:
        steps.append(f"critic={critic_decision}")
    if state.get("repair_used"):
        steps.append(f"repair={state.get('repair_decision') or 'used'}")
    if (state.get("repair_retrieval_count") or 0) > 0:
        steps.append(f"repair_retrievals={state.get('repair_retrieval_count', 0)}")
    if state.get("drift_triggered"):
        steps.append("drift_triggered")
    pending = state.get("drift_pending_scrapes") or []
    if pending:
        steps.append(f"pending_scrapes={len(pending)}")
    if state.get("ingestion_just_ran"):
        steps.append("ingested_new_filing")
    return " -> ".join(steps)


In [16]:
# ---------------------------------------------------------------------------
# Pipeline runners — all now accept the global index, no per-question corpus
# ---------------------------------------------------------------------------

def run_simple_rag(question: str, index: CorpusIndex) -> Dict[str, Any]:
    start = time.time()
    retrieved = index.dense_search(question, top_k=CONFIG["rerank_top_k"])
    few_shot_examples = select_few_shot_examples(question)
    gen_chunks, gen_chars = get_model_aware_context_limits(
        "generator", "generator", CONFIG["generator_max_context_chunks"], CONFIG["generator_max_context_chars"]
    )
    context   = format_retrieved_context(
        retrieved,
        max_chunks=gen_chunks,
        max_chars=gen_chars,
    )
    raw       = safe_invoke_llm(
        "generator", generator_prompt, {"question": question, "context": context, "few_shot_examples": few_shot_examples}, task_name="generator"
    )
    answer    = normalize_answer_text(raw)
    return {
        "pipeline": "simple_rag",
        "rewritten_query": question,
        "retrieved_chunks": retrieved,
        "retrieved_doc_names": extract_retrieved_doc_names(retrieved),
        "context_retries": 0, "context_eval_relevant": None,
        "generated_answer": answer,
        "critic_decision": None, "repair_used": False, "repair_decision": None,
        "final_answer": answer, "latency_sec": time.time() - start,
        "needs_decomposition": False, "repair_retrieval_count": 0,
    }


def run_advanced_rag(question: str, index: CorpusIndex) -> Dict[str, Any]:
    start = time.time()
    retrieved = index.hybrid_search(
        question,
        bm25_top_k=CONFIG["bm25_top_k"],
        dense_top_k=CONFIG["dense_top_k"],
        rerank_top_k=CONFIG["rerank_top_k"],
    )
    few_shot_examples = select_few_shot_examples(question)
    gen_chunks, gen_chars = get_model_aware_context_limits(
        "generator", "generator", CONFIG["generator_max_context_chunks"], CONFIG["generator_max_context_chars"]
    )
    context = format_retrieved_context(
        retrieved,
        max_chunks=gen_chunks,
        max_chars=gen_chars,
    )
    raw     = safe_invoke_llm(
        "generator", generator_prompt, {"question": question, "context": context, "few_shot_examples": few_shot_examples}, task_name="generator"
    )
    answer  = normalize_answer_text(raw)
    return {
        "pipeline": "advanced_rag",
        "rewritten_query": question,
        "retrieved_chunks": retrieved,
        "retrieved_doc_names": extract_retrieved_doc_names(retrieved),
        "context_retries": 0, "context_eval_relevant": None,
        "generated_answer": answer,
        "critic_decision": None, "repair_used": False, "repair_decision": None,
        "final_answer": answer, "latency_sec": time.time() - start,
        "needs_decomposition": False, "repair_retrieval_count": 0,
    }


def run_agentic_rag(question: str, index: CorpusIndex) -> Dict[str, Any]:
    start = time.time()
    state: GraphState = {
        "question":               question,
        "index":                  index,
        "context_retries":        0,
        "repair_used":            False,
        "repair_retrieval_count": 0,
        "is_repair_retrieval":    False,
        "retrieval_sanity_failed": False,
        "sub_queries":            [],
        "needs_decomposition":    False,
        "drift_pending_scrapes":  [],
        "drift_triggered":        False,
        "ingestion_just_ran":     False,
    }
    out = agentic_graph.invoke(state)
    latency = time.time() - start

    generated = normalize_answer_text(out.get("generated_answer", ""))
    final = normalize_answer_text(out.get("final_answer", generated))
    agent_result = {
        "pipeline":                "agentic_rag",
        "rewritten_query":         normalize_answer_text(out.get("rewritten_query", question)),
        "retrieved_chunks":        out.get("retrieved_chunks", []),
        "retrieved_doc_names":     out.get("retrieved_doc_names", []),
        "context_retries":         out.get("context_retries", 0),
        "context_eval_relevant":   out.get("context_eval_relevant"),
        "generated_answer":        generated,
        "critic_decision":         out.get("critic_decision"),
        "critic_reason":           out.get("critic_reason"),
        "repair_used":             out.get("repair_used", False),
        "repair_decision":         out.get("repair_decision"),
        "repair_reason":           out.get("repair_reason"),
        "repair_retrieval_count":  out.get("repair_retrieval_count", 0),
        "needs_decomposition":     out.get("needs_decomposition", False),
        "retrieval_sanity_failed": out.get("retrieval_sanity_failed", False),
        "drift_triggered":         out.get("drift_triggered", False),
        "drift_pending_scrapes":   out.get("drift_pending_scrapes", []),
        "ingestion_just_ran":      out.get("ingestion_just_ran", False),
        "final_answer":            final,
        "latency_sec":             latency,
    }
    agent_result["agent_trace"] = build_agent_route_trace(agent_result)
    return agent_result


In [17]:
# ---------------------------------------------------------------------------
# Evaluation loop
# ---------------------------------------------------------------------------

def run_all_pipelines(eval_df: pd.DataFrame, index: CorpusIndex) -> pd.DataFrame:
    rows = []
    sleep_sec = CONFIG["inter_question_sleep_sec"]

    for q_idx, (_, row) in enumerate(tqdm(eval_df.iterrows(), total=len(eval_df), desc="Questions")):
        qid              = row["id"]
        question         = row["question"]
        reference_answer = row.get("reference_answer", "")
        company          = row.get("company", "")
        ticker           = row.get("ticker", "")
        form_type        = row.get("form_type", "")
        filing_year      = row.get("filing_year")
        evidence_chunk_id = row.get("evidence_chunk_id", "")
        context_sufficient = row.get("context_sufficient", "")
        question_number  = row.get("question_number")
        question_type    = row.get("question_type", "unknown")
        expected_decision = row.get("expected_decision", "answer")

        pipeline_outputs = [
            run_simple_rag(question, index),
            run_advanced_rag(question, index),
            run_agentic_rag(question, index),
        ]

        for out in pipeline_outputs:
            final_answer     = normalize_answer_text(out.get("final_answer", ""))
            generated_answer = normalize_answer_text(out.get("generated_answer", ""))

            predicted_decision = compute_decision_from_text(final_answer)
            decision_accuracy  = 1.0 if predicted_decision == expected_decision else 0.0

            # Token F1 and numerical accuracy — meaningful only when reference_answer is non-empty
            t_f1    = token_f1_score(final_answer, reference_answer) if reference_answer else None
            num_acc = numerical_accuracy_score(final_answer, reference_answer) if reference_answer else None

            rows.append({
                "id":                     qid,
                "question":               question,
                "question_number":        question_number,
                "company":                company,
                "ticker":                 ticker,
                "form_type":              form_type,
                "filing_year":            filing_year,
                "question_type":          question_type,
                "reference_answer":       reference_answer,
                "evidence_chunk_id":      evidence_chunk_id,
                "context_sufficient":     context_sufficient,
                "expected_decision":      expected_decision,
                "pipeline":               out["pipeline"],
                "rewritten_query":        out.get("rewritten_query", question),
                "retrieved_doc_names":    out.get("retrieved_doc_names", []),
                "context_retries":        out.get("context_retries", 0),
                "context_eval_relevant":  out.get("context_eval_relevant"),
                "generated_answer":       generated_answer,
                "critic_decision":        out.get("critic_decision"),
                "critic_reason":          out.get("critic_reason"),
                "repair_used":            out.get("repair_used", False),
                "repair_decision":        out.get("repair_decision"),
                "repair_reason":          out.get("repair_reason"),
                "repair_retrieval_count": out.get("repair_retrieval_count", 0),
                "needs_decomposition":    out.get("needs_decomposition", False),
                "retrieval_sanity_failed": out.get("retrieval_sanity_failed", False),
                "drift_triggered":        out.get("drift_triggered", False),
                "drift_pending_scrapes":  out.get("drift_pending_scrapes", []),
                "ingestion_just_ran":     out.get("ingestion_just_ran", False),
                "agent_trace":            out.get("agent_trace", out.get("pipeline", "")),
                "final_answer":           final_answer,
                "latency_sec":            out.get("latency_sec"),
                "token_f1":               t_f1,
                "numerical_accuracy":     num_acc,
                "predicted_decision":     predicted_decision,
                "decision_accuracy":      decision_accuracy,
                # Filled in by LLM judge later
                "llm_correctness_score":  None,
                "llm_correctness_reason": None,
                "llm_claims_covered":     None,
                "llm_faithfulness_score": None,
                "llm_faithfulness_reason":None,
                "retrieved_chunks":       out.get("retrieved_chunks", []),
            })

        if q_idx < len(eval_df) - 1:
            time.sleep(sleep_sec)

    return pd.DataFrame(rows)


results_df = run_all_pipelines(eval_df, global_index)

Questions:   0%|          | 0/80 [00:00<?, ?it/s]

  [Planner] single | 1 sub-query
  [Planner] single | 1 sub-query
  [Planner] single | 1 sub-query
  [WARN] ContextEvalOutput attempt 1/3 on llama-3.1-8b-instant failed: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.1-8b-instant` in organization `org_01kkg2w4p0fsgaf83q4ph4f28h` service tier `on_demand` on tokens per day (TPD): Limit 500000, Used 499958, Requested 746. Please try again in 2m1.6512s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
  [WARN] Switching agent model from llama-3.1-8b-instant to qwen/qwen3-32b after rate limit.
  [WARN] LLM call attempt 1/3 on llama-3.1-8b-instant failed: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.1-8b-instant` in organization `org_01kkg2w4p0fsgaf83q4ph4f28h` service tier `on_demand` on tokens per day (TPD): Limit 500000, Used 499905, Requested 3522. Please try again in 9m52.1856s. Ne

In [18]:
# ---------------------------------------------------------------------------
# LLM judging (faithfulness + correctness on a sample per pipeline)
# ---------------------------------------------------------------------------

def llm_judge_correctness(question: str, reference_answer: str, candidate_answer: str) -> Tuple[float, float, str]:
    result = safe_invoke_structured(
        "judge", JudgeOutput, judge_correctness_prompt,
        {"question": question, "reference_answer": reference_answer, "candidate_answer": candidate_answer},
        JudgeOutput(score=0.0, claims_covered=0.0, reason="Judge fallback"),
        task_name="judge",
    )
    return max(0.0, min(1.0, float(result.score))), max(0.0, min(1.0, float(result.claims_covered))), result.reason


def llm_judge_faithfulness(question: str, context: str, candidate_answer: str) -> Tuple[float, float, str]:
    result = safe_invoke_structured(
        "judge", JudgeOutput, judge_faithfulness_prompt,
        {"question": question, "context": context, "candidate_answer": candidate_answer},
        JudgeOutput(score=0.0, claims_covered=0.0, reason="Judge fallback"),
        task_name="judge",
    )
    return max(0.0, min(1.0, float(result.score))), max(0.0, min(1.0, float(result.claims_covered))), result.reason


def run_llm_judging(
    results_df: pd.DataFrame,
    sample_n: int = CONFIG["judge_sample_n"],
    random_state: int = CONFIG["random_seed"],
) -> pd.DataFrame:
    if not CONFIG.get("use_llm_judge", False) or sample_n == 0:
        print("Skipping LLM judge; using gold QA metrics only.")
        return results_df.copy()

    df = results_df.copy()
    rng = random.Random(random_state)

    for pipeline in df["pipeline"].unique():
        pipe_idx = df[df["pipeline"] == pipeline].index.tolist()
        sampled  = rng.sample(pipe_idx, min(sample_n, len(pipe_idx)))
        print(f"Judging {len(sampled)} questions for [{pipeline}]…")

        for idx in tqdm(sampled, desc=pipeline, leave=False):
            row = df.loc[idx]
            reference = row["reference_answer"]

            # Correctness (only meaningful when we have a reference answer)
            if reference:
                c_score, c_claims, c_reason = llm_judge_correctness(
                    row["question"], reference, row["final_answer"]
                )
                df.at[idx, "llm_correctness_score"]  = c_score
                df.at[idx, "llm_correctness_reason"] = c_reason
                df.at[idx, "llm_claims_covered"]     = c_claims

            # Faithfulness (always run — uses retrieved context)
            chunks = row.get("retrieved_chunks") if "retrieved_chunks" in df.columns else []
            judge_chunks, judge_chars = get_model_aware_context_limits(
                "judge", "judge", CONFIG["judge_max_context_chunks"], CONFIG["judge_max_context_chars"]
            )
            context_str = format_retrieved_context(
                chunks,
                max_chunks=judge_chunks,
                max_chars=judge_chars,
            ) if chunks and isinstance(chunks, list) and len(chunks) > 0 else row["final_answer"]
            f_score, _, f_reason = llm_judge_faithfulness(
                row["question"], context_str, row["final_answer"]
            )
            df.at[idx, "llm_faithfulness_score"]  = f_score
            df.at[idx, "llm_faithfulness_reason"] = f_reason

            time.sleep(CONFIG["inter_question_sleep_sec"])

    return df


if CONFIG.get("use_llm_judge", False):
    results_df = run_llm_judging(results_df)
else:
    print("Skipping LLM judge; using gold QA metrics only.")


def build_results_input_block(
    results_df: pd.DataFrame,
    pipeline_name: str = "agentic_rag",
) -> pd.DataFrame:
    agentic_df = results_df[results_df["pipeline"] == pipeline_name].copy()
    if agentic_df.empty:
        raise ValueError(f"No rows found for pipeline={pipeline_name!r}")

    export_rows = []
    for _, row in agentic_df.sort_values("question_number").iterrows():
        retrieval_precision, retrieval_recall, retrieval_relevancy = retrieval_metrics(
            row.get("retrieved_chunks", []),
            evidence_chunk_id=row.get("evidence_chunk_id", ""),
            ticker=row.get("ticker", ""),
            filing_year=row.get("filing_year"),
            form_type=row.get("form_type", ""),
        )
        token_precision, token_recall = token_precision_recall(
            row.get("final_answer", ""), row.get("reference_answer", "")
        )

        context_str = format_retrieved_context(
            row.get("retrieved_chunks", []),
            max_chunks=CONFIG["judge_max_context_chunks"],
            max_chars=CONFIG["judge_max_context_chars"],
        ) if row.get("retrieved_chunks") else row.get("final_answer", "")

        faithfulness_score = row.get("llm_faithfulness_score")
        if faithfulness_score is None or pd.isna(faithfulness_score):
            faithfulness_score, _, _ = llm_judge_faithfulness(
                row["question"], context_str, row.get("final_answer", "")
            )

        if row.get("expected_decision", "answer") == "answer" and row.get("reference_answer", ""):
            answer_relevancy, claims_covered, _ = llm_judge_correctness(
                row["question"], row.get("reference_answer", ""), row.get("final_answer", "")
            )
        else:
            answer_relevancy = float(row.get("decision_accuracy", 0.0) or 0.0)
            claims_covered = answer_relevancy

        hallucination = max(0.0, 1.0 - float(faithfulness_score))
        task_completion = float(row.get("decision_accuracy", 0.0) or 0.0)
        correct_answer = correct_answer_binary(
            row.get("expected_decision", "answer"),
            answer_relevancy,
            row.get("numerical_accuracy"),
            row.get("token_f1"),
            task_completion,
        )

        export_rows.append({
            "question_number":      int(row.get("question_number")),
            "retrieval_precision":  float(retrieval_precision),
            "retrieval_recall":     float(retrieval_recall),
            "retrieval_relevancy":  float(retrieval_relevancy),
            "faithfulness":         float(faithfulness_score),
            "answer_relevancy":     float(answer_relevancy),
            "hallucination":        float(hallucination),
            "token_f1":             float(row.get("token_f1") or 0.0),
            "token_precision":      float(token_precision),
            "token_recall":         float(token_recall),
            "task_completion":      float(task_completion),
            "correct_answer":       int(correct_answer),
        })

    return pd.DataFrame(export_rows)


def write_results_input_block(
    metrics_df: pd.DataFrame,
    excel_path: str,
    sheet_name: str = "Results Input",
    start_row: int = 204,
    system_name: str = "Adv RAG + LangGraph",
) -> None:
    from openpyxl import load_workbook

    wb = load_workbook(excel_path)
    ws = wb[sheet_name]
    col_map = {
        "H": "retrieval_precision",
        "I": "retrieval_recall",
        "J": "retrieval_relevancy",
        "K": "faithfulness",
        "L": "answer_relevancy",
        "M": "hallucination",
        "N": "token_f1",
        "O": "token_precision",
        "P": "token_recall",
        "Q": "task_completion",
        "R": "correct_answer",
    }

    for _, row in metrics_df.iterrows():
        target_row = start_row + int(row["question_number"]) - 1
        if ws.cell(target_row, 2).value != system_name:
            raise ValueError(
                f"Row {target_row} does not match system '{system_name}'. Found {ws.cell(target_row, 2).value!r}"
            )
        for col_letter, metric_name in col_map.items():
            ws[f"{col_letter}{target_row}"] = row[metric_name]

    wb.save(excel_path)
    print(f"Wrote {len(metrics_df)} rows to {excel_path} [{sheet_name}!H{start_row}:R{start_row + len(metrics_df) - 1}]")

Skipping LLM judge; using gold QA metrics only.


In [19]:
# ---------------------------------------------------------------------------
# Metrics aggregation + display
# ---------------------------------------------------------------------------

def nanmean(series) -> float:
    vals = [v for v in series if pd.notna(v)]
    return float(np.mean(vals)) if vals else float("nan")


def rate_of(series, match_val) -> float:
    vals = [v == match_val for v in series if pd.notna(v)]
    return float(np.mean(vals)) if vals else float("nan")


def aggregate_metrics(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for pipeline, grp in df.groupby("pipeline"):
        agentic = grp if pipeline == "agentic_rag" else None
        rows.append({
            "pipeline":                  pipeline,
            "n_questions":               len(grp),
            "token_f1":                  nanmean(grp["token_f1"]),
            "numerical_accuracy":        nanmean(grp["numerical_accuracy"]),
            "decision_accuracy":         nanmean(grp["decision_accuracy"]),
            "avg_latency_sec":           nanmean(grp["latency_sec"]),
            "llm_faithfulness_score":    nanmean(grp["llm_faithfulness_score"]),
            "llm_correctness_score":     nanmean(grp["llm_correctness_score"]),
            "llm_claims_covered":        nanmean(grp["llm_claims_covered"]),
            # Agentic-only metrics
            "decomposition_rate":        nanmean(grp["needs_decomposition"]) if "needs_decomposition" in grp else float("nan"),
            "critic_repair_rate":        rate_of(grp["critic_decision"], "repair"),
            "critic_insufficient_rate":  rate_of(grp["critic_decision"], "insufficient_data"),
            "repair_invocation_rate":    nanmean(grp["repair_used"]),
            "avg_context_retries":       nanmean(grp["context_retries"]),
        })
    return pd.DataFrame(rows).set_index("pipeline")


summary_df = aggregate_metrics(results_df)

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.3f}".format)
print("\n=== Summary metrics by pipeline ===")
display(summary_df.T)

# Show agentic-only per-question detail
agentic_rows = results_df[results_df["pipeline"] == "agentic_rag"].copy()
display_cols = [
    "id", "question_type", "question", "reference_answer",
    "generated_answer", "final_answer", "rewritten_query",
    "needs_decomposition", "context_retries", "critic_decision", "critic_reason",
    "repair_used", "repair_decision", "repair_reason", "repair_retrieval_count",
    "retrieval_sanity_failed", "drift_triggered", "ingestion_just_ran",
    "drift_pending_scrapes", "agent_trace",
]
print("\n=== Agentic RAG — per-question detail (LLM answer vs gold answer + route trace) ===")
display(agentic_rows[display_cols].reset_index(drop=True))

if CONFIG.get("auto_export_results_input", False):
    metrics_export_df = build_results_input_block(results_df, pipeline_name="agentic_rag")
    write_results_input_block(
        metrics_export_df,
        CONFIG["results_input_excel_path"],
        sheet_name="Results Input",
        start_row=204,
        system_name="Adv RAG + LangGraph",
    )


=== Summary metrics by pipeline ===


pipeline,advanced_rag,agentic_rag,simple_rag
n_questions,80.000,80.000,80.000
token_f1,0.045,0.055,0.052
numerical_accuracy,0.467,0.383,0.417
decision_accuracy,0.750,0.512,0.750
avg_latency_sec,33.150,92.707,470.372
llm_faithfulness_score,NaN,NaN,NaN
llm_correctness_score,NaN,NaN,NaN
llm_claims_covered,NaN,NaN,NaN
decomposition_rate,0.000,0.225,0.000
critic_repair_rate,NaN,0.032,NaN



=== Agentic RAG — per-question detail (LLM answer vs gold answer + route trace) ===


,id,question_type,question,reference_answer,generated_answer,final_answer,rewritten_query,needs_decomposition,context_retries,critic_decision,critic_reason,repair_used,repair_decision,repair_reason,repair_retrieval_count,retrieval_sanity_failed,drift_triggered,ingestion_just_ran,drift_pending_scrapes,agent_trace
0,SEC_006,extractive,What was Nvidia's gross margin percentage in f...,"Nvidia's gross margin was 72.7% in FY2024, up ...",Nvidia's gross margin percentage in fiscal yea...,Nvidia's gross margin percentage in fiscal yea...,gross margin NVDA filing year 2024,False,0,repair,"The answer has a fixable error: wrong period, ...",True,accept,Model returned a revised answer,0,False,False,False,[],planner -> retrieve -> generate -> critic=repa...
1,SEC_007,extractive,How many employees did Nvidia report at the en...,"As of the end of fiscal year 2025, Nvidia had ...",The answer to the question is not explicitly s...,"Unfortunately, the provided context does not e...",employees NVDA filing year 2025,False,1,repair,"The answer used incorrect period, the question...",True,accept,Model returned a revised answer,0,False,False,False,[],planner -> retrieve -> context_retry=1 -> gene...
2,SEC_008,extractive,What was JPMorgan's CET1 capital ratio at the ...,JPMorgan's Standardized and Advanced CET1 capi...,"<think>\nOkay, let's tackle this question: ""Wh...",Insufficient data: The required information co...,JPM Common Equity Tier 1 (CET1) Capital 2023,False,0,insufficient_data,No reason provided,False,NaN,NaN,0,False,False,False,[],planner -> retrieve -> generate -> critic=insu...
3,SEC_009,extractive,Which federal regulator supervises JPMorgan Ch...,JPMorgan Chase is supervised and regulated by ...,"<think>\nOkay, let's tackle this question: ""Wh...",Insufficient data: The required information co...,JPM item ITEM 1.01. Entry into a Material Defi...,False,1,insufficient_data,No reason provided,False,NaN,NaN,0,False,False,False,[],planner -> retrieve -> context_retry=1 -> gene...
4,SEC_010,extractive,What were Bank of America's Total liabilities ...,Bank of America's total liabilities as of Sept...,"<think>\nOkay, let's see. The user is asking f...",Insufficient data: The required information co...,t.totalLiab quarterlyfilings q JOIN data c ON ...,False,1,insufficient_data,The retrieved context does not explicitly stat...,False,NaN,NaN,0,False,False,False,[],planner -> retrieve -> context_retry=1 -> gene...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,SEC_096,adversarial,What internal AI system does Chevron use for d...,,"<think>\nOkay, let's see. The user is asking w...",Insufficient data: The required information co...,CVX * name Chevron filing year 2023 descriptio...,True,0,NaN,The retrieved context does not mention any spe...,False,NaN,NaN,0,False,True,False,"[(JPM, 2023, 10-K), (JPM, 2024, 10-K)]",planner -> decompose -> retrieve -> generate -...
76,SEC_097,adversarial,What percentage of Walmart employees work remo...,,"<think>\nOkay, let's see. The user is asking w...",Insufficient data: The required information co...,WMT CAST(SUM(remote workers) AS FLOAT) * 100 /...,False,0,NaN,No reason provided,False,NaN,NaN,0,False,True,False,"[(JPM, 2023, 10-K), (JPM, 2024, 10-K)]",planner -> retrieve -> generate -> drift_trigg...
77,SEC_098,adversarial,What internal robot efficiency metrics does Wa...,,"<think>\nOkay, let's see. The user is asking a...",Insufficient data: The required information co...,WMT * 10-K CONFORMED NAME Walmart Inc. REPORTI...,True,0,NaN,No reason provided,False,NaN,NaN,0,False,True,False,"[(JPM, 2023, 10-K), (JPM, 2024, 10-K)]",planner -> decompose -> retrieve -> generate -...
78,SEC_099,adversarial,What percentage of Costco warehouses are fully...,,"<think>\nOkay, let's see. The user is asking w...",Insufficient data: The required information co...,COST CAST(SUM(CASE WHEN automation level Fully...,False,0,NaN,No reason provided,False,NaN,NaN,0,False,True,False,"[(JPM, 2023, 

PermissionError: [Errno 13] Permission denied: 'C:\\Users\\jeeey\\GenAI-with-LLMs\\LangGraph\\eval_skeleton (LangGraph).xlsx'